### Import Libraries : Exhaustive list, not all being used

In [ ]:
# data manipuation
import pandas as pd
import numpy as np

# panda technical analysis 
import pandas_ta as ta

# data visualization
import matplotlib as mpl

import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10,6)
plt.style.use ='seaborn'
plt.rcParams['font.size'] = 8

#-------------------------------------------------------------------------------------------------------------
# statistical visual representation
import seaborn as sns

import cufflinks as cf                                                        # new package - cufflinks
cf.set_config_file(offline=True, theme='pearl',dimensions=((1000,600)))       # cf.set_config() -- (w,h)

import plotly.express as px
px.defaults.width, px.defaults.height = 1000,600

#-------------------------------------------------------------------------------------------------------------
# Self organizing maps(SOM)
from minisom import MiniSom

#-------------------------------------------------------------------------------------------------------------
# K-means clustering(KMC)
from sklearn.cluster import KMeans

# scikit learn(sklearn) package is used primarily for machine learning
# always remember to perform pre-processing and model selcetion before using model implementation
import sklearn
from sklearn.preprocessing import MinMaxScaler,StandardScaler
from sklearn.model_selection import train_test_split,TimeSeriesSplit,cross_val_score,GridSearchCV,RandomizedSearchCV

from sklearn.pipeline import Pipeline

# sklearn evaluation metrics
from sklearn.metrics import precision_recall_curve, roc_curve,RocCurveDisplay,ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, f1_score, recall_score,precision_score,roc_auc_score,auc
from sklearn.metrics import classification_report, confusion_matrix

# Regression
from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet

# K-means clustering(KMC)
from sklearn.cluster import KMeans

# Support Vector Machine(SVM)
from sklearn.svm import SVC

# DTEM(Decision Trees and Ensemble Models) - Bagging - Random Forest
from sklearn.ensemble import RandomForestClassifier

# K nearest neighbours(KNN)
from sklearn.neighbors import KNeighborsClassifier

# Principal Component Analysis(PCA)
from sklearn.decomposition import PCA

#-------------------------------------------------------------------------------------------------------------
# DTEM(Decision Trees and Ensemble Models) - Boosting - Gradient Boosting(Extreme Gradient Boosting)
from xgboost import XGBClassifier,plot_importance,to_graphviz

#-------------------------------------------------------------------------------------------------------------
# statistical analysis & optimization
import scipy.stats as scs
import scipy.optimize as sco

# statsmodels is a Python package that provides a complement to scipy for statistical computations
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.tsa.statespace.varmax import VARMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, coint, grangercausalitytests,kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

#-------------------------------------------------------------------------------------------------------------
# Shapley values, used in DTEM
import shap

# Pyfolio package used for strategy backtesting
import pyfolio as pf

# to reload another python script 
from importlib import reload

# Ignore warnings - optional
import warnings
warnings.filterwarnings('ignore')

# data retrival
import yfinance as yf
 
# float display
pd.options.display.float_format = '{:0.4f}'.format

# operating system
import os

# Image package
from PIL import Image

# table formation
from tabulate import tabulate

# datetime handling
from datetime import datetime

# Part A : Statistical Analysis from Udemy Course

## 01_exploring data

### loading data

In [ ]:
# using read() method
path = 'C:\\Users\\Admin\\Python - Statistical Analysis\\Loading Datasets\\load.csv'
with open(path, mode ='r') as f1:
    print(f1.read())
    # f1.read()                                                       # read() - not desirable - check output
    

# using readlines() method
data = []
cols = None
with open(path, mode = 'r') as f1:
    for line in f1.readlines():                                       # readlines() - read every line as list element- one list only
        val =line.replace('\n','').split(',')
        if cols is None:
            cols = val
        else:
            data.append(val)
            
df1 = pd.DataFrame(data = data,columns = cols)
            
# using np.loadtxt() method
# create 2d array - takes every row and column
df2 = np.loadtxt(path,skiprows =1,delimiter =',')                      # np.loadtxt()
df2.dtype                                                              # dtype() - type of numpy array 
  
# using np.genfromtxt() method
# create 1d array - 1d array with every row as tuple
df3 = np.genfromtxt(fname = path, delimiter=',', names = True)         # np.genfromtxt()
df3.dtype                                                              # here f8 means 8 bit or 64 bytes float                                     
df3['A'][2:5]    

#--------------------------------------------------------------------------------------------------------------
# condition on multiple columns of dataframe
diabetes = pd.read_csv('C:\\Users\\Admin\\Python - Statistical Analysis\\Exploring Data\\Preparing Datasets\\Diabetes.csv')
diabetes.loc[(diabetes[['Glucose','BMI','Age']] ==0).any(axis=1),:]
OR
diabetes.loc[(diabetes.Glucose == 0) | (diabetes.BMI == 0) | (diabetes.Age == 0) ,:]



### outlier handling

In [ ]:
d1 = np.loadtxt('C:\\Users\\Admin\\Python - Statistical Analysis\\Exploring Data\\Outliers\\outlier_1d.txt')
d2 = np.loadtxt('C:\\Users\\Admin\\Python - Statistical Analysis\\Exploring Data\\Outliers\\outlier_2d.txt')
d3 = np.loadtxt('C:\\Users\\Admin\\Python - Statistical Analysis\\Exploring Data\\Outliers\\outlier_curve.txt')

#--------------------------------------------------------------------------------------------------------------
# 1d array  
mean, sd = np.mean(d1), np.std(d1)
z_score = np.abs((d1-mean)/sd)
threshold = 3
good_point = z_score < threshold
bad_point = z_score > threshold

visual_scatter = np.random.normal(size = len(d1))

plt.figure(figsize =(16,6))
plt.scatter(d1[good_point], visual_scatter[good_point], label = 'good')
plt.scatter(d1[~good_point], visual_scatter[~good_point], label = 'bad')
plt.legend()
plt.show()

#--------------------------------------------------------------------------------------------------------------
# 2d array
mean, cov = np.mean(d2, axis=0), np.cov(d2[:,0], d2[:,1])
good_point = scs.multivariate_normal(mean, cov).pdf(d2) > 0.01/100               # scs.multivariate_normal()

plt.figure(figsize =(16,6))
plt.scatter(d2[good_point,0], d2[good_point,1], label = 'good')
plt.scatter(d2[~good_point,0], d2[~good_point,1], label = 'bad')
plt.legend()
plt.show()

titanic = pd.read_csv(r"C:\Users\Admin\Python Panda\Course_Materials_Part1\Video_Lecture_NBs\titanic.csv")
cov = titanic.cov()
rvs = scs.multivariate_normal.rvs(cov = cov,size =20)   # will create 20 rows each with same elements of cov matrix

#--------------------------------------------------------------------------------------------------------------
# polynomial fitting with outliers removal
# multiple iteraions with deg = 5
x, y = d3[:,0].copy(),d3[:,1].copy()
for i in range(5):
    p = np.polyfit(x,y, deg = 5)
    ys = np.polyval(p,x)
    threshold =3
    good_point = y- ys < threshold
    
    x_bad, y_bad = x[~good_point], y[~good_point]
    x, y = x[good_point], y[good_point]
    plt.figure(figsize=(16,6))
    plt.scatter(x,y, label = 'good data', s=1,c ='green')
    plt.scatter(x_bad,y_bad, label = 'bad points',s=5, c = 'red')
    plt.plot(x,np.polyval(p,x), label = 'polyfit {}'.format(i))
    plt.ylim(0,10)
    plt.legend()
    plt.show()
    
    if((~good_point).sum()) == 0:
        break

### 1D probability distribution

In [ ]:
d1 = np.loadtxt('C:\\Users\\Admin\\Python - Statistical Analysis\\Exploring Data\\1D Distribution\\example_1.txt')
d2 = np.loadtxt('C:\\Users\\Admin\\Python - Statistical Analysis\\Exploring Data\\1D Distribution\\example_2.txt')

#--------------------------------------------------------------------------------------------------------------
# histogram
# linewidth,linestyle,histtype
bins = np.linspace(start = min(np.min(d1),np.min(d2)), stop = max(np.max(d1),np.max(d2)),num =50 )
plt.figure(figsize = (16,6))
counts1, bins, _ = plt.hist(d1, bins =bins, label = 'd1',histtype = 'step',lw=3)         # here '_' means everything
plt.hist(d2, bins =bins,label ='d2', histtype = 'step',linewidth= 2,linestyle = ':')     # linewidth, linestyle, histtype               
plt.legend()
plt.show()

#--------------------------------------------------------------------------------------------------------------
# bee swarmplot
values = np.concatenate([d1,d2])                                                         # np.concatenate()
types = np.concatenate([np.ones(d1.shape), np.zeros(d2.shape)])                          # np.ones()
dataset = pd.DataFrame({'values': values, 'types': types} )

plt.figure(figsize = (12,8))
sns.swarmplot(x = 'types', y= 'values', data = dataset)                                  # sns.swarmplot()

#--------------------------------------------------------------------------------------------------------------
# boxplot - shows all quartiles -0,25,50,75,100 and outliers
# whis- by deafult 1.5, Proportion of the IQR(inter quartile range) past the low and high quartiles to extend the plot whiskers
plt.figure(figsize = (12,8))
sns.swarmplot(x = 'types', y= 'values', data = dataset, color = 'red', alpha = 0.3)    
sns.boxplot(x = 'types', y= 'values', data = dataset, whis = 2.0)                       # sns.boxplot()
   
#--------------------------------------------------------------------------------------------------------------
# violionplot - shows all quartiles -0,25,50,75,100 and outliers
# inner -
plt.figure(figsize = (12,8))
sns.swarmplot(x = 'types', y= 'values', data = dataset, color = 'red', alpha = 0.3)
sns.violinplot(x = 'types', y= 'values', data = dataset, inner = 'quartile')            # sns.violinplot

#--------------------------------------------------------------------------------------------------------------
# 1d empirical cdf
sd1,sd2 = np.sort(d1),np.sort(d2)
cdf1, cdf2 = np.linspace(1/d1.size,1,d1.size),np.linspace(1/d2.size,1,d2.size)

plt.figure(figsize =(12,8))
plt.plot(sd1,cdf1, label = 'd1 ecdf')
plt.plot(sd2,cdf2, label= 'd2 ecdf')
plt.hist(sd1,density = True,bins =50,histtype = 'step', alpha =0.5)
plt.hist(sd2,density = True,bins=50,histtype = 'step', alpha =0.5)
plt.legend()
plt.show()


### ND probability distribution

In [ ]:
df2 = pd.read_csv('C:\\Users\\Admin\\Python - Statistical Analysis\\Exploring Data\\ND Distribution\\height_weight.csv')

#--------------------------------------------------------------------------------------------------------------
# 2d histogram
plt.hist2d(x = df2['height'], y= df2['weight'], bins=20,cmap = 'viridis')                                  # plt.hist2d()

#--------------------------------------------------------------------------------------------------------------
# contour plot
# np.histogram gives hist-frequency, x_edge-x value for each edge surface, y_edge -y value for each edge surface

hist, x_edge, y_edge = np.histogram2d(x = df2['height'], y= df2['weight'], bins=20)                        # np.histogram2d()
x_center = 0.5*(x_edge[1:] + x_edge[:-1])
y_center = 0.5*(y_edge[1:] + y_edge[:-1])
plt.contour(x_center,y_center,hist, cmap = 'viridis', levels = 5)                                          # plt.contour()
plt.hist2d(x = df2['height'], y= df2['weight'], bins=20,cmap = 'magma', alpha =0.5)                        
plt.show()

#--------------------------------------------------------------------------------------------------------------
# kde plot- converts 2d histogram to gaussian surface
# here for sns plot, panda series has been passed for x and y values
sns.kdeplot(x = df2['height'], y= df2['weight'], cmap = 'viridis')                                         # sns.kdeplot()
plt.hist2d(x = df2['height'], y= df2['weight'], bins=20,cmap = 'magma', alpha =0.5)                        
plt.show()


## 02_characterization

In [ ]:
path = 'C:\\Users\\Admin\\Python - Statistical Analysis\\Characterising\\dataset.txt'

dataset = np.loadtxt(path)
scs.mode(dataset)                                    # scs.mode() - mode(i.e. mean, mode, median) of the array

#--------------------------------------------------------------------------------------------------------------

mean_ds = np.mean(dataset)
std_ds = np.std(dataset)
skewness = scs.skew(dataset)

# normal(gaussian) approximation to dataset distribution
xs = np.linspace(np.min(dataset), np.max(dataset), 100)
ys1 = scs.norm.pdf(xs,loc = mean_ds, scale = std_ds)                         # scs.norm.pdf()         

# normal approximation to dataset distribution with skewness
# you can check from below graph - not a good fit because skewness changes mean & std of normal distribution
# skew normal distribution is a continuous probability distribution that generalises the normal distribution to allow for non-zero skewness

ys2 = scs.skewnorm.pdf(xs,skewness,loc = mean_ds, scale = std_ds)            # scs.skewnorm.pdf()

# with correction of mean, std
ps = scs.skewnorm.fit(dataset)                                               # scs.skewnorm.fit() - gives skewness, mean, std
ys3 = scs.skewnorm.pdf(xs,*ps)                                               # scs.skewnorm.pdf()

plt.hist(dataset, bins =50, density = True,alpha =0.3)
plt.plot(xs,ys1,linestyle = '--', color = 'red',label = 'normal')
plt.plot(xs,ys2,linestyle = ':', color = 'green',label = 'skewnormal')
plt.plot(xs,ys3,linestyle = '-', color = 'yellow',label = 'corrected skewnormal')

plt.legend()
plt.show()

## 03_Probability

### Gaussian probability density function(normal distribution)

In [ ]:
Image.open('C:\\Users\\Admin\\Python - Statistical Analysis\\gaussian.JPG')
# area under all curves = 1 since its cdf

### Uniform probability density function

In [ ]:
Image.open('C:\\Users\\Admin\\Python - Statistical Analysis\\uniform.JPG')

### Binomial probability density function

In [ ]:
Image.open('C:\\Users\\Admin\\Python - Statistical Analysis\\binomial.JPG')
# very similar to gaussian distribution, as n increases --> approach gaussian
# p = probability of the event, read P(k|n,p) as probabilty of k given n and p

### Poission probability density function

In [ ]:
Image.open('C:\\Users\\Admin\\Python - Statistical Analysis\\poission.JPG')
# mu = mean rate of event = how many times event likely to occur within the given time period (# of car crashes in day = 10)

### Exponential probability density function

In [ ]:
Image.open('C:\\Users\\Admin\\Python - Statistical Analysis\\exponential.JPG')
# amount of time clerk spends with customer. The time is known to have exponential distribution with the average time = 4 mins 
# lambda = decay rate of event = 1/4 = 0.25 means this event(meeting with customer) decays at rate 0.25 customer/min
# probability of time i.e probability that clerk spends 5 mins,6 mins etc.
# in poission, we take frequency of event whereas in exponential, we take time unit

### lognormal distribution

In [ ]:
Image.open('C:\\Users\\Admin\\Python - Statistical Analysis\\lognormal.JPG')
# probability density function given mean, std and shape(here mean and std = 0,1)
# useful when variable takes only positive values

### Student t-distribution

In [ ]:
Image.open('C:\\Users\\Admin\\Python - Statistical Analysis\\t-dist.JPG')
# probability density function given mean, std and degree of freedom(here mean and std = 0,1), gamma function embedded
# used when no of observations is small

### discrete probability distributions(binom, poission) pmf

In [ ]:
xs = np.linspace(-5,10,200)
ks = np.arange(50)

#--------------------------------------------------------------------------------------------------------------
# pmf - probability mass function, similar to pdf (continuous probability distributions)

pmf_binom = scs.binom.pmf(ks,n= 50, p= 0.25)                       # scs.binom.pmf()
pmf_poission = scs.poisson.pmf(ks,mu=30)                           # scs.poission.pmf()

plt.bar(x = ks, height = pmf_binom, label = 'binom')               # plt.bar()
plt.bar(x = ks, height = pmf_poission, label = 'poission')
plt.legend()
plt.show()

### continuous probability distributions- pdf

In [ ]:
# pdf, cdf, ppf, rvs

xs = np.linspace(-5,10,200)

pdf_normal = scs.norm.pdf(xs,loc =5,scale = 2)                # scs.normal.pdf()
pdf_uniform = scs.uniform.pdf(xs,loc =-4,scale = 10)          # scs.uniform.pdf()
pdf_exponential = scs.expon.pdf(xs, loc =-2, scale =2)        # scs.expon.pdf(), lambda = 1/scale, loc just to shift graph 
pdf_lognormal = scs.lognorm.pdf(xs,s=1)                       # scs.lognorm.pdf()
pdf_studentt = scs.t.pdf(xs,df=1)                             # scs.t.pdf()

scs.norm.sf()                                                 # scs.norm.sf() - opposite of cdf

plt.figure(figsize =(16,6))
plt.plot(xs,pdf_uniform, label = 'uniform')
plt.plot(xs,pdf_normal, label = 'normal')
plt.plot(xs,pdf_exponential, label = 'exponential')
plt.plot(xs,pdf_studentt, label = 'student t')
plt.plot(xs,pdf_lognormal, label = 'lognormal')
plt.legend();

#--------------------------------------------------------------------------------------------------------------
# To shift and/or scale the distribution use the loc and scale parameters
# Note that shifting the location of a distribution does not make it a “noncentral” distribution
pdf_studentt_1 = scs.t.pdf(xs,df=1, loc=4, scale =2)
pdf_studentt_2 = scs.t.pdf((xs-4)/2,df=1, loc=0, scale=1)
plt.figure(figsize =(16,6))
plt.plot(xs,pdf_studentt_1, label = 'student t 1')
plt.plot(xs,pdf_studentt_2, label = 'student t 2')
plt.legend();



### Interpolation/Extrapolation techniques

**interp1d is considered legacy API and is not recommended for use in new code. Consider using more specific interpolators instead.**

### Piecewise linear interpolation

In [ ]:
x = np.linspace(0, 10, num=11)
y = np.cos(-x**2 / 9.0)
xnew = np.linspace(0, 10, num=1001)
ynew = np.interp(x = xnew, xp = x, fp = y)                                       # interp()

plt.plot(x, ynew, '-', label='linear interp')
plt.plot(x, y, 'o', label='data')
plt.legend(loc='best')
plt.show()

### Cubic splines

- Of course, piecewise linear interpolation produces corners at data points, where linear pieces join. To produce a smoother curve, you can use cubic splines, where the interpolating curve is made of cubic pieces with matching first and second derivatives. In code, these objects are represented via the CubicSpline class instances. An instance is constructed with the x and y arrays of data, and then it can be evaluated using the target xnew values:

- Cubic spline interpolation is a way of finding a curve that connects data points with a degree of three or less. Splines are polynomial that are smooth and continuous across a given plot and also continuous first and second derivatives where they join
- Cubic splines are only piecewise continuous, meaning that a sufficiently high derivative (third) is discontinous

In [ ]:
x = np.linspace(0, 10, num=11)
y = np.cos(-x**2 / 9.)
spl = CubicSpline(x, y)                                                     # CubicSpline()

fig, ax = plt.subplots(4, 1, figsize=(5, 7))
xnew = np.linspace(0, 10, num=1001)
ax[0].plot(xnew, spl(xnew))
ax[0].plot(x, y, 'o', label='data')
ax[1].plot(xnew, spl(xnew, nu=1), '--', label='1st derivative')             # spl(nu=1)     
ax[2].plot(xnew, spl(xnew, nu=2), '--', label='2nd derivative')             # spl(nu=2)
ax[3].plot(xnew, spl(xnew, nu=3), '--', label='3rd derivative')             # spl(nu=3)
for j in range(4):
    ax[j].legend(loc='best')
plt.tight_layout()
plt.show()

### Optimization using scipy

Let us suppose that a merry farmer has 75 roods (4 roods = 1 acre) on which to plant two crops: wheat and corn. To produce these crops, it costs the farmer (for seed, water, fertilizer, etc. ) $\$120$ per rood for the wheat, and $\$210$ per rood for the corn. The farmer has $\$15,000$ available for expenses, but after the harvest the farmer must store the crops while awaiting favorable or good market conditions. The farmer has storage space for 4,000 bushels. Each rood yields an average of 110 bushels of wheat or 30 bushels of corn. If the net profit per bushel of wheat (after all the expenses) is $\$1.30$ and for corn is $\$2.00$, how should the merry farmer plant the 75 roods to maximize profit?

It is not obvious what to do. Wheat is cheaper to plant, and yields more per rood, but it is less profitable.

Let x be the number of roods of wheat planted, and y be the number of roods of corn planted. The profit function is: P=(110)(1.3)x+(30)(2)y=143x+60y

There are some constraint inequalities, specified by the limits on expenses, storage and roodage. They are:

120x+ 210y<=15000 (The total amount spent cannot exceed the amount the farm has)

110x+30y<=4000 (The amount generated should not exceed storage space.)

x+y<=75 (We cannot plant more space than we have.)

0<=x and 0<=y(all amounts of planted land must be positive.)

To solve this problem, we cast it as minimization problem, which minimizes a function f(X) subject to some constraints. We create a proxy function for the negative of profit, which we seek to minimize.

Note we use inequality constraints here. You could put some as equality, e.g. if you want to make sure you spend all $15000, or if you want to ensure you fill up your storage. It could be a mistake to require both of those though, as it may be infeasible.

In [ ]:
def c1(X):
    x, y = X
    return -(120*x + 210*y -15000)              # this has to be non-negative

def c2(X):
    x, y = X
    return -(110*x + 30*y -4000)  

def c3(X):
    x, y = X
    return -(x + y -75)  

def c4(X):
    x, y = X
    return x  

def c5(X):
    x, y = X
    return y 

def profit(X):
    x, y = X
    return -(143*x + 60*y)

sco.minimize(fun = profit, x0=[10,20],constraints = [{'type':'ineq', 'fun':f} for f in [c1,c2,c3,c4,c5]])

### Empirical distribution

In [ ]:
# here we want to calculate pdf and cdf

from scipy.interpolate import interp1d

xs = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0]
ys = [0.2, 0.165, 0.167, 0.166, 0.154, 0.134, 0.117, 0.108, 0.092, 0.06, 0.031, 0.028, 0.048, 0.077, 0.103, 0.119, 0.119, 0.103, 0.074, 0.038, 0.003]

# here xs is random variable value and ys is pdf

x = np.linspace(min(xs), max(xs), 1000)
y1 = interp1d(xs,ys, kind = 'linear')(x)                                       # scipy.interpolate.interp1d()
pdf = y2 = interp1d(xs,ys, kind ='quadratic')(x)                               # see how function is passed

plt.scatter(xs,ys,label = 'data',c = 'red')
plt.plot(x,y1, label = 'Linear interpolation')
plt.plot(x,y2, label = 'quardratic interpolation')
plt.legend();


Using the `interp1d` we can now find a probability value for any `x` value.

How can we calculate the CDF and the probability we would find a value between two bounds? Using `scipy.integrate`. Scipy to the rescue once again!

We have many options:

* `scipy.integrate.trapz` for low accuracy but high speed. Accuracy scales as `O(h)` - trapaziod method - linear interpolation between data points
* `scipy.integrate.simps` for medium accuracy and pretty high speed. Accuracy scales as `O(h^2)`- simpson method -quadratic interpolation between data points
* `scipy.integrate.quad` for high accuracy and low sped. Arbitrary accuracy.

There are a few more functions, look them up if you're curious.

In [ ]:
from scipy.integrate import simpson

# this function calculates area between 2 points a and b
def get_area(xs,ys,a,b):
    x = np.linspace(a,b,1000)
    y = interp1d(xs,ys, kind = 'quadratic')(x)    
    return simpson(y,x)                                           # scipy.integrate.simpson()

# this function calculates cdf
def get_cdf(xs,ys,v):
    return get_area(xs,ys,min(xs),v)

cdf = [get_cdf(xs,ys,i) for i in np.linspace(min(xs),max(xs), 100)]


### Area Calcualtion using integrate method

### Sampling distribution

In [ ]:
Image.open('C:\\Users\\Admin\\Python - Statistical Analysis\\sampling distribution.JPG')

#### Rejection Sampling

Let us say we don't have a nice easy analytic distribution, and that we cannot use one to approximate our distribution. We can brute force our sampling by sampling the uniform distribution and just throwing away points. It works like this:

1. Sample a uniform `x` value
2. Sample a uniform `y` value from `0` to the maximum probability in your PDF
3. If $y > p(x)$, throw out the point.

Easier to see in practise. Lets try and sample from the *unnormalised* distribution $p(x) = \sin(x^2) + 1$ from $0 \rightarrow 4$ 

In [ ]:
# x values are 0 --> 4 and y values are 0 --> 2
# here distribution means probability distribution
# this is unnormalized distribution because cdf <> 1 (cdf = area under pdf from 0 --> 4) 
# here pdf = p(x) means funciton value i.e.if we pass 1000 values for x how y values are coming
# loc and scale are not necessirly mean and std. for uniform its a and b respectively.

def func_ps(x):
    return np.sin(x**2)+1                                               # np.sin()

n = 100000
random_x = scs.uniform.rvs(loc = 0, scale=4,size =n)                    # scs.uniform.rvs()
random_y = scs.uniform.rvs(loc = 0, scale=2,size =n)
passed = random_y <= func_ps(random_x)

xs = np.linspace(0,4,1000)
ps = func_ps(xs)

# to calculate pdf - y value/(ydx)-- similar to histogram pdf calculation
# converting unnormalized distribution to normalized distribution
pdf = ps/simpson(ps,xs)


# empirical means based on observations
plt.figure(figsize =(12,8))
plt.hist(random_x[passed], density = True,bins=100,label = 'sampled distribution- rejection based')   
#plt.scatter(random_x[passed], random_y[passed],label = 'passed', c ='green')
#plt.scatter(random_x[~passed], random_y[~passed],label = 'not passed', c ='red')
plt.plot(xs,pdf, label = 'empirical pdf distribution')
#plt.plot(xs,ps, label = 'function')
plt.legend();


#### Inversion Sampling

This is harder to conceptually understand. But the way I can put it the simplest is that we know that, for all PDFs, the CDF is going to go from 0 to 1. If we can uniformly sample the CDF from 0 to 1, can we invert our function so that we can recover the $x$ value that gives the sampled CDF value? For some functions, yes. For some, no, the math isn't solvable.

The function above is tricky, so let's find a function that's simpler to sample.

Let's say our new PDF is $p(x) = 3 x^2$ from $0 \rightarrow 1$. As opposed to the previous function, this one is normalised.

So, we can find the CDF via

$$ CDF(x) = \int_0^x p(x^\prime) dx^\prime = x^3 $$

Once we have the CDF, we want to invert it. That is, at the moment, we have an $x$ value and get a $y$ value - the CDF. We want to give it a CDF and get the $x$ value:

$$ y = x^3 \rightarrow x = y^3 $$

Which means $y = x^{1/3}$. Or to put the CDF syntax back in, $x = CDF^{1/3}$

In [ ]:
# this is normalised function

def pdf(x):
    return 3*x**2

def cdf(x):
    return x**3

def icdf(cdf):
    return cdf**(1/3)

xs = np.linspace(0,1,num=100)
pdfs = pdf(xs)
cdfs = cdf(xs)

# fig creates figure object : it creats two boxes and it refers to full graph (inclusing two boxes)
# ax objects are reference to graphs/subplots
fig,ax = plt.subplots(ncols=2,figsize =(12,4))                # plt.subplots()

u_samps = scs.uniform.rvs(size =1000)
x_samps = icdf(u_samps)

ax[0].plot(xs, pdfs, label = 'pdf')
ax[0].hist(x_samps, histtype = 'step',bins =50, density= True, label = 'sampled distribution')
ax[0].legend()
ax[1].plot(xs,cdfs, label = 'cdf')
ax[1].legend();

In [ ]:
# this is unnormalised function, continuation from previous exercise

def func_ps(x):
    return np.sin(x**2) + 1

xs = np.linspace(0,4,1000)
ps = func_ps(xs)
pdfs = ps/simpson(ps,xs)
cdfs = np.cumsum(pdfs)/np.sum(pdfs)                            # np.cumsum()

u_samps = scs.uniform.rvs(size =1000)
x_samps = interp1d(cdfs,xs)(u_samps)

fig, axes = plt.subplots(ncols=2,figsize =(12,4))               
axes[0].plot(xs, pdfs, label = 'pdf')
axes[0].hist(x_samps, histtype = 'step',bins =100, density= True, label = 'sampled distribution- inversion sampling')
axes[0].legend()
axes[1].plot(xs,cdfs, label = 'cdf')
axes[1].legend();

### Central Limit theorom

The central limit theorem (CLT) states that the distribution of sample means approximates a normal distribution as the sample size gets larger, regardless of the population's distribution.Sample sizes equal to or greater than 30 are often considered sufficient for the CLT to hold.

The distribution of means has standard deviation proportional to the underlying distribution divided by the root of the number of samples.

Or another way to say this, if you have $N$ samples, the mean of your samples is distributed as per a normal around the true mean, with standard deviation $\sigma/\sqrt{N}$. 

Or *another* way of saying this, is that if you go from $N_1$ data points to $N_2$ data points, you can determine the mean $\sqrt{N_2/N_1}$ more accurately. 4 times as many samples doesn't give 4 times more accuracy, only double the accuracy.

In [ ]:
def get_data(n):
    data = np.concatenate([scs.expon.rvs(size = n//2,scale=1),scs.skewnorm.rvs(5,size =n//2,loc=2, scale=3)])
    np.random.shuffle(data)
    return data
                          

In [ ]:
sample_size =[10,50,100,500,1000,5000,10000]

std_list = []
for n in sample_size:
    std_dev = np.std([get_data(n).mean() for i in range(1000)])
    std_list.append(std_dev)
    
plt.scatter(sample_size,std_list)
plt.plot(sample_size,1/np.sqrt(sample_size),c ='red')


## 04_Hypothesis Testing

In [ ]:
#--------------------------------------------------------------------------------------------------------------
# a die is thrown 500 times and count of getitng 6 is 98
# Null hypothesis --> p(6) = 1/6, means fair die
data = np.loadtxt("C:\\Users\\Admin\\Python Statistical Analysis\\Hypothesis Testing\\Loaded_Die\\loaded_500.txt")
returns,counts = np.unique(data,return_counts=True)

num_sixes = (data == 6).sum()
num_total = data.size
n = np.arange(1, num_total)
n_prob = scs.binom.pmf(n,num_total,1/6)

plt.plot(n,n_prob, label = "binom dis")
plt.axvline(x =num_sixes,color = "red",ls = "dotted" ,label = "obs value")
plt.axvline(x =num_total/6,color = "green",ls = ":" ,label = "mean")
plt.legend();


### asteriod testing

In [ ]:
# NASA has helpfully provided a dataset which lists recorded meteorite impacts  
# Can we utilise this dataset to predict the chance that, within 1000 years, a high-impact meteor will strike the planet?
# Let's define high-impact as an asteroid greater than 1km in diameter. 

df_meteorite = pd.read_csv(r"C:\Users\Admin\Python Statistical Analysis\Hypothesis Testing\Meteorites\Meteorite_Landings.csv")

df2_meteorite = df_meteorite.dropna(subset = ["year", "fall"])
df2_meteorite = df2_meteorite.loc[df_meteorite.mass > 0]

# Now, mass is a positive value so it'll probably be easier to look at log mass instead of mass itself.

masses = df2_meteorite["mass"]
logmass = np.log(masses)

ms = np.linspace(-5,20,100)

mean, std = np.mean(logmass), np.std(logmass)
pdf_norm = scs.norm.pdf(ms,loc = mean,scale = std)

# either you give parameters for pdf/cdf/ppf/sf/rvs(above) such as loc, scale, df, shape etc.
# OR
# use observations data to calculate parameter which contains all the info about the distribution
# for both the approaches, you need to know about distribution i.e normal,lognormal,skewnormal,t,chi-square etc.

p_lognorm = scs.lognorm.fit(logmass,)                              # scs.lognorm.fit()
pdf_lognorm = scs.lognorm.pdf(ms,*p_lognorm)                      # scs.lognorm.pdf() 

p_skewnorm = scs.skewnorm.fit(logmass)                            # scs.skewnorm.fit()
pdf_skewnorm = scs.skewnorm.pdf(ms,*p_skewnorm)                   # scs.skewnorm.pdf()

# So either the log-norm or skew-norm looks like an adequate fit to the data. We want to extrapolate this distribution out to a mass we're concerned about
# we have taken lognormal distribution

plt.figure(figsize=(12,8))
plt.hist(logmass, bins = 100, density = True, alpha = 0.5)
plt.plot(ms,pdf_norm, label = "normal")
plt.plot(ms,pdf_lognorm, label = "lognormal", color = "red", ls = ":")
plt.plot(ms,pdf_skewnorm, label = "skewnormal", ls = "--")
plt.show()

# the mass of a 1 km diameter meteor.

mass_of_doom = np.log((4/3) * np.pi * (500**3) * 1600 * 1000)  # Just using a spherical approximation and some avg density



So we have here the probability of an asteroid being above a certain mass when it hits Earth. But to answer the question "What is the probability that one or more asteroids of high mass strike Earth in 1000 years?" we need to factor in the actual time component. Assume that in the next 1000 years, we predict to have $N$ impacts.

$$P(>=1\  {\rm highmass}) = 1 - P(0\ {\rm highmass}) = 1 - P(N\ {\rm not\_highmass}) = 1 - P({\rm not\_highmass})^N$$

Imagine a similar question: Prob of getting no sixes in 5 rolls. Well its $$\frac{5}{6}\times\frac{5}{6}\times\frac{5}{6}\times\frac{5}{6}\times\frac{5}{6} = \left(\frac{5}{6}\right)^5 $$
The prob of getting one or more sixes is then $1 - (5/6)^5$.

So to give a number, we need to calculate $N$ from the yearly rate, number of years, and our detection efficiency and use that with the probability that any given impact is not high mass.


In [ ]:
no_events_year = df2_meteorite.year.value_counts().max()
frac_obs_events = 0.2 
total_events = no_events_year * (1000/frac_obs_events)
prob_small_mass = scs.lognorm.logcdf(mass_of_doom,*p_lognorm)                  # scs.lognorm.logcdf() - we have taken log probability and later exp(logProb) == prob
prob_small_event = np.exp(prob_small_mass*total_events)
prob_big_event = 1 - prob_small_event
print("probablity of big event is {:0.2f} %".format(prob_big_event*100))


### Election rigging testing - proportion testing

In [ ]:
# here we are using proportion testing : we are converting binomial distribution  to normal distribution assuming large population size
# assumption- population size is at least 20 times sample size i.e you ask 100 people in street whom they voted
# for its applicatiom, you must have at least 10 sucess and 10 failure counts i.e. at least 10 people votes for A and 10 against A
# sigma = sqrt(p*(1-p)/N) , p = proportion of success (here % votes in favour of A) , N = sample size

# The final results for the distrct are 39% Candidate A, 61% Candidate B.
# You call a a hundred members of the general public who reported as voting. 48 out of the 100 reported voting for Candidate A.
# Null Hypothesis - 39% of votes went to Candidate A 
# Alternative Hypothesis - > 39% of votes went to Candidate A 

In [ ]:
hypothesized_value = 0.39
sample_statistics = 0.48
sigma = np.sqrt(hypothesized_value*(1-hypothesized_value)/100)
std_error = sigma
test_statistics = (sample_statistics - hypothesized_value)/(std_error)


# one tailed test at 5% level os significance

def test_results(test_statistics,significance_level,one_tailed = True):
    
    if one_tailed:
        critical_value = scs.norm.ppf(1-significance_level)
        
        if test_statistics > critical_value:
            print("critical value : {:0.2f} and test statistics : {:0.2f} we can reject null hypothesis".format(critical_value,test_statistics))
        else:
            print("critical value : {:0.2f} and test statistics : {:0.2f} we cant reject null hypothesis".format(critical_value,test_statistics))
    else:
        critical_value_r = scs.norm.ppf(1-significance_level/2)
        critical_value_l = scs.norm.ppf(significance_level/2)
        
        if (test_statistics > critical_value_r) | (test_statistics < critical_value_l):
            print("critical value left : {:0.2f},critical value right : {:0.2f} and test statistics : {:0.2f} we can reject null hypothesis".format(critical_value_l,critical_value_r,test_statistics))
        else:
            print("critical value left : {:0.2f},critical value right : {:0.2f} and test statistics : {:0.2f} we cant reject null hypothesis".format(critical_value_l,critical_value_r,test_statistics))
 
        
     

## 05_Conclusion

###  Pearson Correlation test

In [ ]:
# Null hypothesis = the frequency of junk food doesnt cause acne i.e correlation is low
# here we will use pearson correlation test

data =pd.read_csv(r"C:\Users\Admin\Python Statistical Analysis\Conclusion\junkfood.csv")
foods = data.Food.unique()
alpha = 0.05

for f in foods:
    df2 = data.loc[data.Food == f,:]
    test_statistics, p_value = scs.pearsonr(df2["Acne"], df2["Frequency"])
    if p_value < alpha:
        print("Significant correlection between food {} frequency and Acne, p value is {}".format(f,p_value))
    else:
         print("No significant correlection between food frequency and Acne")

###  Gaussian Process

In the above pseudo-GP process we wanted to interpolate 20 points (well 18 'unknown' points given the ends are fixed), and so we did this by drawing from a 20-D Gaussian. But obviously there are an infinite number of points we might want to interpolate to, so we'd draw from an infinite-dimensional Gaussian. This is what "non-parametric" means - infinite dimensions.

And by drawing over and over we can come up with some mean prediction and some standard deviation on it. Note that this is dependent on our choice for covariance. The cov we have used has only one parameter, its length scale. The larger $l$ gets the smoother and less changing the predictions become. This number that we 'pick' is known as a hyperparameter. And its something we have to fit for when creating a real GP. And in terms of nomenclature, moving from a MVGaussian to a Gaussian process we dont have a mean, we have a mean function (because we can ask for the mean at any x). Similarly the covariance matrix depends on the input data points (in our previous example we didnt have any, just hack fixed the first and last point) and the points we want to evaluate at, so we have a covariance function. The shape of the covariance (ie we had squared distance) is called the kernel (you can have different functions, squared distance is just a useful one)

In [ ]:
Image.open(r"C:\Users\Admin\Python Statistical Analysis\bivariate_Gaussian.PNG")

# used viridis as colormap, can see where probability density is high

In [ ]:
length_scale = 20
x = np.arange(length_scale)                                       
y = np.atleast_2d(x).T                               # np.atleast_2d()                              

length = 50
cov = np.exp((-1/length)*((x - y)**2))               # -----> calculating distance between two points
sns.heatmap(cov)                                     # sns.heatmap()

size = 200 
rvs = scs.multivariate_normal.rvs(size = size,cov=cov)    # scs.mutivariate_normal.rvs()
rvs -= np.linspace(rvs[:,0], rvs[:,-1],x.size).T


mean = np.mean(rvs,axis = 0)
std = np.std(rvs,axis = 0)

plt.figure(figsize =(16,10))
plt.plot(x,mean,marker = "X")
plt.fill_between(x,y1 = (mean + std), y2 = (mean - std), color = "black", alpha =0.2)

for i in range(size):
    plt.plot(x,rvs[i,:],alpha =0.5, ls = "-",)

### Cosmic Impacts

In [ ]:
# you have to try and characterise incoming particle detections as either cosmic rays or as stellar ejecta 
# For this test, you only have one particle to classify, which impacted with an energy of 1200 MeV.
# To help you out, you have a dataset containing stellar ejecta and cosmic rays, located in cosmic_data.txt. 
# It contains two columns, the first, the energy of the impact in MeV, and the second the probability it was from solar ejecta.

data = np.loadtxt(r"C:\Users\Admin\Python Statistical Analysis\Conclusion\Cosmic Impacts\cosmic_data.txt")

steller = data[data[:,1] > 0.9,0]
cosmic  = data[data[:,1] < 0.1,0]
test = 1200

# to remove contanimation

cosmic = cosmic[cosmic > 700]

# here normal distribution doesnt fit well- can be confirmed
# sometimes lognormal gives peaks around 0,can be handled using loc and scale parameters (see below)

params_s = scs.lognorm.fit(steller, loc = 400,scale =100)
params_c = scs.lognorm.fit(cosmic,loc = 1300,scale =100)

xs = np.linspace(0,1500,1000)

pdf_s = scs.lognorm.pdf(xs,*params_s)
pdf_c = scs.lognorm.pdf(xs,*params_c)

plt.figure(figsize =(12,8))
plt.hist(data[:,0], bins =100,alpha =0.2,label = "all", density = True)
plt.hist(steller, bins =100, histtype="step",label = "steller data",density = True)
plt.hist(cosmic, bins =100, histtype="step",label = "cosmic data",density = True)
plt.plot(xs,pdf_s,label = "steller model")
plt.plot(xs,pdf_c,label = "cosmic model")
plt.legend()

prob_cosmic = scs.lognorm.pdf(test, *params_c) * cosmic.shape[0]
prob_stellar = scs.lognorm.pdf(test, *params_s) * steller.shape[0]
total_prob = prob_cosmic + prob_stellar

final = prob_cosmic / total_prob

print("probability of ejecta coming from cosmic is {} %".format(final*100))

### Emission Standards

New vehicles have to pass emission standards, which have been constantly tightening. Your job is to take the testing report and determine whether or not the vehicle passes the test, and with what confidence.

It says that the maximum CO content is 1.0g/kg. Emission data samples are located in `emission_small.txt` and `emission_large.txt`. We'll analyse both datasets independently.



In [ ]:
data_small = np.loadtxt(r"C:\Users\Admin\Python Statistical Analysis\Hypothesis Testing\Car Emission\emission_small.txt")
data_large = np.loadtxt(r"C:\Users\Admin\Python Statistical Analysis\Hypothesis Testing\Car Emission\emission_large.txt")

### Using the small dataset - t distribution/t-test

In [ ]:
sample_statistics = 1

# for hypothesis testing, generate hypothetical distribution of pdf to calculate p value and compare with alpha

sample_mean = np.mean(data_small)
sample_std = np.std(data_small)/np.sqrt(data_small.size)

xs = np.linspace(0.5,1.5,100)
p_value = 1 - scs.t.cdf(sample_statistics, df = 9, loc = mean, scale = std )        # scs.t.cdf() - 10 observations so df = n-1

This isn't good for the company! At the moment, we do **not** have confidence that their new vehicle passes the emission standard. We should ask for more data.  We can reject the hypothesis that we have confidence their vehicle passes the standard.

### Large Dataset - normal distribution/z test

With the larger dataset, we have enough points that we can use the normal approximation (as the degrees of freedom increases, Student-T approaches a normal). >30 datapoints is a good estimate.

In [ ]:
sample_statistics = 1

# for hypothesis testing, generate hypothetical distribution of pdf to calculate p value and compare with alpha

sample_mean = np.mean(data_large)
sample_std = np.std(data_large)/np.sqrt(data_large.size)

xs = np.linspace(0.5,1.5,100)
p_value = 1 - scs.norm.cdf(sample_statistics,loc = mean, scale = std )   

### Diagnosing Diabetes

In [ ]:
# basically we want to predict whether patient has diabetes or not based on some parameters (Glucose, BMI) based on csv data

path = r"\Users\Admin\Python Statistical Analysis\Hypothesis Testing\Diabetes\Diabetes.csv"
df = pd.read_csv(path)

# finding and visualising correlation

df.corr()
plt.figure(figsize =(12,8))
sns.heatmap(df.corr(),annot= True,cmap = 'Reds', fmt ='0.2f')

# we have taken factors with high correlation with outcome
# Look at the histograms for Glucose and BMI - the spikes at zero need to be removed, also dropping Age to make things simpler
df1 = df.loc[:,["Glucose","BMI","Age","Outcome"]]
df2 = df1.loc[(df1.Glucose > 1) & (df1.BMI > 1) ,["Glucose","BMI","Outcome"]]

df2_y = df2.loc[df2.Outcome == 1,:]
df2_n = df2.loc[df2.Outcome == 0,:]

plt.figure(figsize =(12,8))
plt.scatter(x = df2_y["Glucose"], y = df2_y["BMI"], label = "Has Diabetes", c = "red")
plt.scatter(x = df2_n["Glucose"], y = df2_n["BMI"], label = "No Diabetes", c = "green")
plt.xlabel("Glucose", fontsize =18)
plt.ylabel("BMI", fontsize =18)
plt.legend();

So its not perfect, but we can probably do an alright job approximating both these distributions as Gaussians.

Let's also add into the mix a test patient that comes in, with Glucose of 140 and BMI of 35. What is his chance of being diagnosed?

In [ ]:
test_point = [140,35]
prob_test = []

x,y = np.meshgrid(np.linspace(50,200,20), np.linspace(10,50,20), indexing="xy")       # np.meshgrid()
points = np.dstack([x,y])                                                             # np.dstack()

In [ ]:
plt.figure(figsize =(16,10))
for d, l in zip([df2_y,df2_n],[True,False]):
    mean = d.loc[:,["Glucose","BMI"]].mean(axis =0)
    cov = d.loc[:,["Glucose","BMI"]].cov()
    pdf = mn.pdf(points,mean = mean,cov = cov)                         # mn.pdf() - requires (Glucose,BMI) as a datapoint
    prob_test.append(mn.pdf(test_point,mean = mean,cov = cov))
    plt.contour(x,y,pdf, cmap = "viridis" if l == True else "magma")
    plt.scatter(d["Glucose"], d["BMI"], c = "red" if l == True else "green")
plt.axvline(x = 140, ls = "--")
plt.axhline( y=35, ls ="--")
plt.xlabel("Glucose", fontsize =18)
plt.ylabel("BMI", fontsize =18)
plt.show()

prob_test
prob = df2_y.shape[0] * prob_test[0]/(df2_y.shape[0] * prob_test[0] + df2_n.shape[0] * prob_test[1])
print("patient has {:0.2f}  % probability of diabetes".format(prob*100))

# Part B : Basic Statistics

## P-value Testing Framework 

- For 2 tailed t test with 10% significance level, the p-value threshold is 10%, not 5%.
- Example Set up <br>
Significance level: α=0.10
Two-tailed test <br>a
Sample size: n=16 <br>
Degrees of freedom: df=n−1=15 <br>
Since it's two-tailed :α/2=0.05
We need the critical value: $t_{0.95,15}$ 

- Step 1: Critical t-value <br>
From t-tables (or software):
t0.95,15≈1.753
So rejection region is:
$t < -1.753 \quad \text{or} \quad t > 1.753$

- Step 2:<br>
Assume your calculated test statistic is: t=1.85
Since: 1.85 > 1.753
👉 It falls in rejection region.
So we reject H0 at 10% level.

- Step 3: p-value approach <br>
Now let’s check via p-value.
For t=1.85, df=15:
One-tailed probability ≈ 0.041
Two-tailed p-value:
p=2×0.041=0.082
Now compare:
0.082<0.10
✅ Reject H0

----------------------------------------------------------------------------------------------------------------

- significance/confidence - threshold on probability at which point, we may reject the null hypothesis
- null hypothesis - hypothesis you want to reject
- p value - probability that null hypothesis is correct
- critical value/p value - lowest level of significane at which we can reject null hypothesis, generally 0.05 or 5%
- p value of 3.4% mean we can reject at 4% or more but not at lower percentage
- to calculate p-value : create hypothetical distribution and check where sample-statistics lies in that distribtion and area to the right
- to test hypothesis either use critical value OR p-value

In [ ]:
Image.open(r"C:\Users\Admin\Python Statistical Analysis\standard_error.PNG")

## Homoscedasticity & Heteroscedasticity

### Basic definition about Homoscedasticity & Heteroscedasticity


(1) **Homoskedasticity**

- this means the error variance is constant across observations: $Var(\epsilon_t)=\sigma^2$
- So, whether the market is calm or volatile, the model assumes the same unconditional variance.
- For example: $\epsilon_t \sim (0,\sigma^2)$ There is no systematic change in volatility over time.


(2) **Conditional Heteroscedasticity**

- Conditional heteroscedasticity occurs when the variance of the error depends on past information(past squared returns, past variance, past shocks)
- This is quite important in financial time series, variance at time t depends on info at t-1, $\operatorname{Var}(\epsilon_t \mid \mathcal{F}_{t-1}) = \sigma_t^2$
- Where $\sigma_t^2$ changes over time based on information available up to $t-1$.
- For a time series:$\operatorname{Var}(\epsilon_t\mid\mathcal{F}_{t-1})\neq \sigma^2$, where $\mathcal{F}_{t-1}$ represents all information available up to time $t-1$.
- one such example is GARCH model.

(3) **Unconditional Heteroscedasticity**

- Unconditional heteroscedasticity occurs when the variance changes systematically with observable explanatory variables:
- variance of error term/residual is not constant and we can't model it using past information (past squared returns, past variance, past shocks) because it changes systematically with observable explanatory variables.
- $\operatorname{Var}(\epsilon_i\mid X_i)=\sigma^2h(X_i)$
- where $h(X_i)=1+\alpha_1X_{1i}+\alpha_2X_{2i}+\cdots+\alpha_kX_{ki}$
  
    Therefore, $\operatorname{Var}(\epsilon_i\mid X_i)=\sigma^2\left(1+\alpha_1X_{1i}+\alpha_2X_{2i}+\cdots+\alpha_kX_{ki}\right)$

    If $\alpha_1=\alpha_2=\cdots=\alpha_k=0$ then $h(X_i)=1$ and therefore, $\operatorname{Var}(\epsilon_i\mid X_i)=\sigma^2$ which implies **homoscedasticity**.
- Here, the variance depends on $X_i$. This type of heteroscedasticity is commonly tested using tests such as the **Breusch–Pagan test**.

(4) **Financial Example**

- Financial returns commonly exhibit volatility clustering.This means: large movements tend to be followed by large movements and small movements tend to be followed by small movements
- For example: $\text{large shock}\rightarrow\text{large volatility}\rightarrow\text{large shock}$
- $\text{calm period}\rightarrow\text{low volatility}\rightarrow\text{calm period}$
- This is an important example of **conditional heteroscedasticity**.

(5) **why shock is replacing return**

- $\epsilon_t$ = shock/innovation are error terms but in finance, they are essentially same as returns after removing the mean $r_t = \mu + \epsilon_t$, where $\mu$ = 0 in daily data.
- Shock is unpredictable part of your series. 

(5) **ARCH Model**

- A simple ARCH(1) model can be written as: $\operatorname{Var}(\epsilon_t\mid\mathcal{F}_{t-1})=\sigma_t^2=\omega+\alpha_1\epsilon_{t-1}^2$
- The current conditional variance therefore depends on the previous period's squared shock.
- More generally, an ARCH($q$) model is: $\sigma_t^2=\omega+\sum_{i=1}^{q}\alpha_i\epsilon_{t-i}^2$


(6) **GARCH Model**

- A general GARCH($p,q$) model is: $\sigma_t^2=\omega+\sum_{i=1}^{q}\alpha_i\epsilon_{t-i}^2+\sum_{j=1}^{p}\beta_j\sigma_{t-j}^2$
- The model allows current volatility to depend on:past squared shocks, past conditional variances
- For example, GARCH(1,1): $\sigma_t^2=\omega+\alpha_1\epsilon_{t-1}^2+\beta_1\sigma_{t-1}^2$

(7) **Important Distinction: Unconditional vs Conditional Heteroscedasticity**

| Feature | Unconditional Heteroscedasticity | Conditional Heteroscedasticity |
|---|---|---|
| Main idea | Variance depends on $X$ | Variance depends on past information |
| Typical setting | Cross-sectional regression | Time-series / financial returns |
| Variance | $\operatorname{Var}(\epsilon_i\mid X_i)=\sigma^2h(X_i)$ | $\operatorname{Var}(\epsilon_t\mid\mathcal{F}_{t-1})=\sigma_t^2$ |
| Example | Salary variance changes with income/education | Stock volatility changes over time |
| Typical model/test | Breusch–Pagan | ARCH/GARCH |
| Key driver | Explanatory variables | Past shocks / volatility |

### Tests for Conditional and Unconditional heteroscedasticity

(1) **Breusch–Pagan Test**

- Purpose : The **Breusch–Pagan test** is used to test for **unconditional heteroscedasticity**, where the variance of the regression error depends on the explanatory variables $X$.
- Hypotheses The null hypothesis is: $H_0:\operatorname{Var}(\epsilon_i\mid X_i)=\sigma^2$ This represents **homoscedasticity**, meaning that the error terms/residuals have constant conditional variance.
- The alternative hypothesis is: $H_1:\operatorname{Var}(\epsilon_i\mid X_i)=\sigma^2 h(X_i)$ This represents **heteroscedasticity**, where the variance of the error terms depends on the explanatory variables/regressors.


(2) **Variance Function**
- A common specification is: $h(X_i)=1+\alpha_1X_{1i}+\alpha_2X_{2i}+\cdots+\alpha_kX_{ki}$ Therefore, $\operatorname{Var}(\epsilon_i\mid X_i)=\sigma^2\left(1+\alpha_1X_{1i}+\alpha_2X_{2i}+\cdots+\alpha_kX_{ki}\right)$

- If $\alpha_1=\alpha_2=\cdots=\alpha_k=0$ then $h(X_i)=1$ and therefore, $\operatorname{Var}(\epsilon_i\mid X_i)=\sigma^2$ which implies **homoscedasticity**.
- The function $h(X_i)$ describes how the variance changes with the explanatory variables.


(3) **Cross-Sectional Data**

Cross-sectional data means data collected on many different units at approximately the same point in time.
For example, Suppose in 2025 we collect data on salary, education and work experience for 1,000 people. There is no time-series dimension for each individual.


(4) **Why Breusch–Pagan Test?**

- Consider the standard linear regression: $Y_i=\beta_0+\beta_1X_i+\epsilon_i$ 
- OLS commonly assumes: $\operatorname{Var}(\epsilon_i\mid X_i)=\sigma^2$ The Breusch–Pagan test checks whether this constant-variance assumption is violated.


(5) **Tests for Conditional Heteroscedasticity**

The **ARCH-LM test**, also known as the **Engle test**, checks for ARCH effects.
- The idea is to regress squared residuals on their own lags:
- $\hat{\epsilon}_t^2=\alpha_0+\alpha_1\hat{\epsilon}_{t-1}^2+\alpha_2\hat{\epsilon}_{t-2}^2+\cdots+\alpha_q\hat{\epsilon}_{t-q}^2+u_t$
- The null hypothesis is:$H_0:\alpha_1=\alpha_2=\cdots=\alpha_q=0$
- Under $H_0$, the variance does not depend on past squared errors.
- The alternative hypothesis is: $H_1:\text{At least one }\alpha_i\neq0$ which indicates ARCH effects.

**Ljung–Box Test on Squared Residuals**

The Ljung–Box test can also be applied to squared residuals:$\hat{\epsilon}_t^2$

If squared residuals exhibit significant autocorrelation, this provides evidence of time-dependent volatility.

## Regression Model

### Popular Regression Model

Suppose the true population regression model is
$Y_i = \beta_0 + \beta_1 X_{i1} + \beta_2 X_{i2} + \cdots + \beta_k X_{ik} + \epsilon_i$

where:
- $\beta_0$ = population intercept
- $\beta_1,\ldots,\beta_k$ = population slope coefficients
- $\epsilon_i$ = random error term
- $k$ = number of explanatory variables

The population assumption for the error variance is $\operatorname{Var}(\epsilon_i \mid X) = \sigma^2$
Here, $\sigma^2$ is the **true but unknown population error variance**.

### OLS Error Variance: Population vs. Sample Estimate

(1) **Basic definition**

- $\operatorname{Var}(\epsilon_i) = \sigma^2$ this is population variance but the problem is this is unknown
- The error for observation $i$ is $ \epsilon_i = Y_i - E(Y_i \mid X_i)$
- The assumption $\operatorname{Var}(\epsilon_i \mid X) = \sigma^2$ means that the errors have the **same variance (same level of uncertainty or noise)** for all observations i.e 
    you take 50 observations or 100 observations..... you get the same variance of error term/residual.
- It does **not** mean that every error equals $\sigma^2$.

(2) **Sample Regression**

- In practice, the true population coefficients are unknown, so we estimate them from a sample: $ \hat{\beta}_0,\hat{\beta}_1,\ldots,\hat{\beta}_k $
- The fitted regression is $\hat{Y}_i=\hat{\beta}_0+\hat{\beta}_1X_{i1}+\cdots+\hat{\beta}_kX_{ik}$ We then calculate the **residual**:
- $\hat{\epsilon}_i = Y_i - \hat{Y}_i$
- An important distinction is $\hat{\epsilon}_i \neq \epsilon_i$ The true error $\epsilon_i$ is unobservable, whereas the residual $\hat{\epsilon}_i$ can be calculated from the sample.

(3) **Estimating the Population Error Variance**

- Since the true errors $\epsilon_i$ are unknown, we cannot directly calculate $\sigma^2$. Instead, we use the sample residuals.
- $\sigma^2\neq\hat{\sigma}^2$
- First calculate the **Sum of Squared Errors (SSE)**:$SSE = \sum_{i=1}^{n}\hat{\epsilon}_i^2$
- The unbiased estimator of the population error variance is $\hat{\sigma}^2 = \frac{SSE}{n-k-1}$ This quantity is also called the **Mean Squared Error (MSE)**:
- $MSE = \frac{SSE}{n-k-1}$ Therefore, $\hat{\sigma}^2 = MSE$

(4) **Why Do We Divide by $n-k-1$?**

- Suppose we have $n = \text{number of observations}$ and $k$ explanatory variables.
- The regression also contains an intercept $\beta_0$. Therefore, the total number of estimated parameters is $k+1$.
- These parameters consume $k+1$ degrees of freedom. Hence, the residual degrees of freedom are
- $df_{\text{error}} = n-(k+1) = n-k-1$ Therefore, $\hat{\sigma}^2 =\frac{SSE}{n-k-1}$
- $\hat{\sigma}^2=\frac{SSE}{n-k-1}=MSE=\text{estimated error variance}$


(5) **Standard Error of an Individual Coefficient**

- For coefficient $\hat{\beta}_j$, the variance is the $j$-th diagonal element of the variance-covariance matrix:
    $\operatorname{Var}(\hat{\beta}_j\mid X)=\sigma^2\left[(X^\top X)^{-1}\right]_{jj}$
- Therefore, the estimated variance is $\widehat{\operatorname{Var}}(\hat{\beta}_j\mid X)= \hat{\sigma}^2 \left[(X^\top X)^{-1}\right]_{jj}$
- The standard error is the square root of the estimated variance: $SE(\hat{\beta}_j)=\sqrt{\hat{\sigma}^2\left[(X^\top X)^{-1}\right]_{jj}}$
- Substituting the MSE, $SE(\hat{\beta}_j)=\sqrt{\frac{SSE}{n-k-1}\left[(X^\top X)^{-1}\right]_{jj}}$

(6) **Complete Flow**

The complete relationship can be summarized as
- Step 1: Population error variance $\operatorname{Var}(\epsilon_i\mid X)=\sigma^2$ where $\sigma^2$ is the true but unknown population error variance.
- Step 2: Calculate residuals $ \hat{\epsilon}_i=Y_i-\hat{Y}_i$
- Step 3: Calculate SSE $ SSE= \sum_{i=1}^{n}\hat{\epsilon}_i^2$
- Step 4: Estimate error variance $\hat{\sigma}^2 = MSE = \frac{SSE}{n-k-1}$
- Step 5: Estimate coefficient variance $ \widehat{\operatorname{Var}}(\hat{\beta})= \hat{\sigma}^2(X^\top X)^{-1}$
- Step 6: Calculate standard error $ SE(\hat{\beta}_j) = \sqrt{\widehat{\operatorname{Var}}(\hat{\beta})}= \sqrt{\hat{\sigma}^2 \left[(X^\top X)^{-1}\right]_{jj}}$


(7) **Population Error vs Estimated Residual**

- The population model is: $Y_i=\beta_0+\beta_1X_i+\epsilon_i$
- The true error is therefore: $\epsilon_i=Y_i-\beta_0-\beta_1X_i$
- However, $\beta_0$ and $\beta_1$ are unknown.
- After estimating the model using sample data: $\hat{Y}_i=\hat{\beta}_0+\hat{\beta}_1X_i$
- The estimated residual is: $ \hat{\epsilon}_i=Y_i-\hat{Y}_i$
- Therefore: $\epsilon_i\neq\hat{\epsilon}_i$ in general.
 Variance of the Error Term

(8) **variance of the error**

- The variance of the population error is:$ \operatorname{Var}(\epsilon_i)=E[\epsilon_i^2]=\sigma^2$ assuming: $E[\epsilon_i]=0$
- The estimated error variance is: $\hat{\sigma}^2 =\frac{\displaystyle\sum_{i=1}^{n}\hat{\epsilon}_i^2}{n-k-1}$

where:

- $n$ = number of observations
- $k$ = number of explanatory variables
- $k+1$ = number of estimated parameters, including the intercept

- The numerator is the **Sum of Squared Errors (SSE)**: $SSE=\sum_{i=1}^{n}\hat{\epsilon}_i^2$

Therefore:

$\hat{\sigma}^2=\frac{SSE}{n-k-1}$, This is also called the **Residual Mean Square** or **MSE**:

$MSE=\frac{SSE}{n-k-1}$

Hence: $\hat{\sigma}^2=MSE$

## De-mystify covariance Matrix

### Variance - Covariance Matrix construction

(1) **Covariance Matrix**

Let $X \in \mathbb{R}^{n \times p}$
where:
- $p$ = number of features / variables, 
- $n$ = number of observations, 
- Each row represents one observation / asset,
- Each column represents one feature / variable

(2) **Centre the Data**

- Before calculating the covariance matrix, we first centre the data.
- The centred matrix is $\tilde{X} = X - \mu$, where $\mu$ represents the column-wise mean vector.
- For each variable $j$,$\mu_j = \frac{1}{n}\sum_{i=1}^{n}X_{ij}$ and therefore $\tilde{X}_{ij} = X_{ij} - \mu_j$
- It is important to ensure that $X$ is centred before using the matrix product to calculate covariance.

(3) **Covariance Matrix**

- For centred data, the covariance matrix can be written as $ \Sigma = \frac{1}{n}\tilde{X}^{T}\tilde{X}$  where $\Sigma \in \mathbb{R}^{p\times p}$, 
- The $(j,k)$ element of the covariance matrix is $\Sigma_{jk}=\frac{1}{n}\sum_{i=1}^{n}\tilde{X}_{ij}\tilde{X}_{ik}$
- For the sample covariance matrix, the unbiased estimator generally uses $S = \frac{1}{n-1}\tilde{X}^{T}\tilde{X}$

(4) **The distinction is:**

- $\frac{1}{n}$ → population covariance / maximum-likelihood form under a normal model
- $\frac{1}{n-1}$ → unbiased sample covariance estimator


(5) **Connection with the One-Dimensional Case**

- For one variable $x$, the variance is $\operatorname{Var}(x)=\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2$
- The matrix expression $ \frac{1}{n}\tilde{X}^{T}\tilde{X}$ is simply the multidimensional/generalized version of this formula.

For example, if

$$X =\begin{pmatrix} x_{11} & x_{12} & \cdots & x_{1p}\\ x_{21} & x_{22} & \cdots & x_{2p}\\ \vdots & \vdots & \ddots & \vdots\\ x_{n1} & x_{n2} & \cdots & x_{np} \end{pmatrix} $$

then
$$ X^TX = \begin{pmatrix} \sum x_1^2 & \sum x_1x_2 & \cdots\\ \sum x_2x_1 & \sum x_2^2 & \cdots\\ \vdots & \vdots & \ddots \end{pmatrix} $$

where the summations are taken over the $n$ observations.

Therefore, $$ \tilde{X}^{T}\tilde{X} $$ contains:

- Sum of squares on the diagonal
- Sum of cross-products off the diagonal

After dividing by $n$ (or $n-1$ for the sample covariance), these become:

- Variances on the diagonal
- Covariances on the off-diagonal


### Variance-Covariance Matrix of OLS Coefficients

Consider the OLS regression model
$y = X\beta + \epsilon$

where $X \in \mathbb{R}^{n\times p}$

and the OLS estimator is $\hat{\beta}=(X^TX)^{-1}X^Ty$

- The variance-covariance matrix of the estimated coefficients is $ \operatorname{Var}(\hat{\beta})=\sigma^2(X^TX)^{-1}$
- under the standard OLS assumptions: $E(\epsilon\mid X)=0$ and $ \operatorname{Var}(\epsilon\mid X)= \sigma^2 I_n$


(2) **Interpretation of the Variance-Covariance Matrix**

The variance-covariance matrix has the form $$ \operatorname{Var}(\hat{\beta}) = \begin{pmatrix}\operatorname{Var}(\hat{\beta}_1) & \operatorname{Cov}(\hat{\beta}_1,\hat{\beta}_2)
& \cdots\\ \operatorname{Cov}(\hat{\beta}_2,\hat{\beta}_1)
& \operatorname{Var}(\hat{\beta}_2) & \cdots \\ \vdots & \vdots & \ddots \end{pmatrix} $$

(3) **Diagonal Elements**

- The diagonal elements are the variances of the estimated coefficients: $\operatorname{Var}(\hat{\beta}_j)$
- The standard error of $\hat{\beta}_j$ is therefore $SE(\hat{\beta}_j)=\sqrt{\operatorname{Var}(\hat{\beta}_j)}$
- Thus $ SE(\hat{\beta}_j)= \sqrt{\left[\operatorname{Var}(\hat{\beta})\right]_{jj}}$



(4) **Off-Diagonal Elements**

- The off-diagonal elements are the covariances between estimated coefficients: $\operatorname{Cov}(\hat{\beta}_i,\hat{\beta}_j)$
- These covariance terms are important when studying **multicollinearity** because correlated explanatory variables can result in strong relationships between their estimated coefficients.


(5) **Variance of OLS Coefficients**

- For OLS, $\operatorname{Var}(\hat{\beta})=\sigma^2(X^TX)^{-1}$
- The matrix $(X^TX)^{-1}$ captures the effect of the design matrix $X$ on the uncertainty of the coefficient estimates.
- A larger amount of information in $X$ generally leads to lower coefficient variance.


### OLS & Ridge Regression- Coefficients

In ridge regression, the coefficient estimator is $ \hat{\beta}_{\text{ridge}} =(X^TX+\lambda I)^{-1}X^Ty$

where:
- $\lambda \geq 0$ is the regularization parameter
- $I$ is the $p\times p$ identity matrix

The regularization term $\lambda I $ is added to $X^TX$ to reduce the effect of multicollinearity and stabilize the matrix inversion.


(2) **Variance of Ridge Coefficients**

For fixed $X$ and $\operatorname{Var}(\epsilon\mid X)=\sigma^2 I$

the covariance matrix of the ridge estimator is $\operatorname{Var}(\hat{\beta}_{\text{ridge}})=\sigma^2(X^TX+\lambda I)^{-1}X^TX(X^TX+\lambda I)^{-1}$

- More generally, writing $ A=(X^TX+\lambda I)^{-1}$ gives $ \operatorname{Var}(\hat{\beta}_{\text{ridge}}) =\sigma^2 A X^TX A^T$
- Since $X^TX+\lambda I$ is symmetric, $ A^T=A$ and therefore $ \operatorname{Var}(\hat{\beta}_{\text{ridge}})=\sigma^2(X^TX+\lambda I)^{-1}X^TX(X^TX+\lambda I)^{-1}$


(3) **OLS vs Ridge Regression**

- in OLS $\hat{\beta}_{OLS}=(X^TX)^{-1}X^Ty$ and $\operatorname{Var}(\hat{\beta}_{OLS})=\sigma^2(X^TX)^{-1}$
- in Ridge $\hat{\beta}_{ridge}=(X^TX+\lambda I)^{-1}X^Ty$ and $ \operatorname{Var}(\hat{\beta}_{ridge}) =\sigma^2(X^TX+\lambda I)^{-1}X^TX(X^TX+\lambda I)^{-1}$
- Thus, the variance expression for ridge regression is more complex because of the regularization term $\lambda$.


(4) **Important Note on Ridge Regression**

- Ridge regression introduces **bias** into the coefficient estimates. Therefore, unlike OLS, the ridge estimator is generally biased:
- $E[\hat{\beta}_{ridge}\mid X]\neq\beta$
- The main objective of ridge regression is to trade a small amount of bias for a potentially substantial reduction in variance.
- This is particularly useful when the explanatory variables are highly correlated, i.e. when there is **multicollinearity**.

## T-dist and F-dist in OLS regression

### Basics about t-dist and F-dist

(1) **t-test & F-test diff**

|Test type | Numerator DF | Denominator DF|
|---|---|---|
|Individual t test | — | $n-k-1$|
|Overall F test | $k$ | $n-k-1$|
|Total DF | — | $n-1$|

(2) **Why variance use $n-1$**

- $s^2 = \frac{1}{n-1}\sum (X_i-\bar{X})^2$
- we subtract 1 because, we estimated the sample mean $\bar{X}$ that uses 1 degree of freedom

(3) **t-dist & F-test**

- t-distribution is built from 1 chi-square variable whereas  F-distribution is built from 2 chi-squares variables
- t has 1 degree of freedom parameter F test has 2 degree of freedom parameters
- $ t\text{-statistic} = \frac{z}{\sqrt{\chi^2_{\nu}/\nu}} = \frac{z}{\sqrt{\chi^2(df)/df}}$
  
  where $z$ = standard normal, divided by square root of chi-square gives t distribution, it has only 1 df
- $ F\text{ test statistic}=\frac{\chi^2(k)/k} {\chi^2(n-k-1)/(n-k-1)} = \frac{MSR}{MSE} = \frac{SSR/k}{SSE/(n-k-1)}$

  where ratio of two chi-square, each divided by their df, it has 2 dfs

(4) **summary**
| Distribution | Built from | df parameters |
|---|---|---|
| chi-square | sum of squared normals | 1 (just k) |
| t-distribution | normal / chi-square | 1 ($n-k-1$) |
| F-distribution | chi-square / chi-square | 2 (numerator & denominator) |

### SSE follows chi-square, SSR & SST also

- Recall chi-square distribution $\chi^2 = z_1^2 + z_2^2 + \cdots + z_n^2$, it is sum of squared standard normal variables
- $SSE=\sum_{i=1}^{n}(Y_i-\hat{Y}_i)^2=\sum_{i=1}^{n}\epsilon_i^2$
- In regression we assume errors are normally distributed $\epsilon_i \sim N(0,\sigma^2)$
- when we standardize the residual $\frac{\epsilon_i}{\sigma}\sim N(0,1)$, so $\frac{SSE}{\sigma^2}=\sum_{i=1}^{n}\left(\frac{\epsilon_i}{\sigma}\right)^2=\chi^2(n-k-1)$

(2) **why df = $n-k-1$**

- even though we are summing $n$ squared terms, we lose degrees of freedom because residuals are not free, they are constrained by estimated parameters $\beta_0,\ldots,\beta_k$
- $k$ slopes = $k+1$ ($\beta_1,\ldots,\beta_k$) so df = $n-k-1$


similarly $\frac{SSR}{\sigma^2}\sim\chi^2(k)$

$\frac{SST}{\sigma^2}\sim\chi^2(n-1)$

### t-statistics follows t-distribution

- t-statistic follow t-distribution, not chi-square distribution
- $t=\frac{\hat{\beta}}{SE(\hat{\beta})}$
- numerator follows normal distribution, whereas denominator follows chi-square because SE is built from SSE
- overall ratio = t-distribution

### chi-square, chi & Normal distribution

-  $\sqrt{\chi^2(df)}\sim\text{chi distribution}$ square root of chi-square is not normal distribution, instead it is chi-distribution

| Distribution | Formula |
|---|---|
| chi-square | $z_1^2+z_2^2+\cdots+z_k^2$ $(k\ df)$ |
| chi | $\sqrt{z_1^2+z_2^2+\cdots+z_k^2}$ $(k\ df)$ |
| Normal | just $z$ alone |

- square root of chi-square gives chi distribution, not normal distribution. The only exception is df = 1 where $\sqrt{z^2}=|z|$ which relates back to normal. The t-dist arises from dividing a normal by a chi-distribution.
- $z$ is standard normal variable
- $z^2=\chi^2(1)$ and if we take $k$ independent standard normal variables $z_1^2+z_2^2+\cdots+z_k^2=\chi^2(k)$ where $k=df$

### standard error of t statistic is connected to chi-square


- we know regression errors are normally distributed $\epsilon_i\sim N(0,\sigma^2)$
- **step by step connection**
- step 1 - residuals are normal : $\epsilon_i\sim N(0,\sigma^2)$
- step 2 - standardize the residuals: $\frac{\epsilon_i}{\sigma}\sim N(0,1)$
- step 3 - square & sum them : $\sum_{i=1}^{n} \left(\frac{\epsilon_i}{\sigma}\right)^2 = \frac{SSE}{\sigma^2} \sim \chi^2(n-k-1)$

    This is exactly the chi-square definition - sum of squared standard normals
- step 4 - SSE to MSE : $MSE=\frac{SSE}{n-k-1}=\frac{\sigma^2\chi^2(n-k-1)}{n-k-1}$
- step 5 - MSE is chi-square scaled $ \frac{MSE}{\sigma^2} = \frac{\chi^2(n-k-1)}{n-k-1}$
- step 6 - SE is square root of MSE : $SE(\hat{\beta})=\sqrt{MSE\,(X'X)^{-1}}$ $=\sigma\sqrt{\frac{\chi^2(n-k-1)}{n-k-1}(X'X)^{-1}}$

- note that $\sigma$, $(n-k-1)$ & $(X'X)^{-1}$ are fixed constants, they only scale the standard error without changing the distribution nature. distribution still remains chi-square.
- MSE is chi-square scaled = meaning that

## ANOVA, Error Variance and Standard Errors

### ANOVA Table


The basic ANOVA decomposition is: $ SST=SSR+SSE$

where
- $SST$ = Total Sum of Squares
- $SSR$ = Regression Sum of Squares
- $SSE$ = Error Sum of Squares

**ANOVA Table**

| Source | Sum of Squares | Degrees of Freedom | Mean Square |
|---|---:|---:|---:|
| Regression | $SSR$ | $k$ | $\displaystyle MSR=\frac{SSR}{k}$ |
| Error | $SSE$ | $n-k-1$ | $\displaystyle MSE=\frac{SSE}{n-k-1}$ |
| Total | $SST$ | $n-1$ | — |

**Coefficient of Determination $R^2$**

- The coefficient of determination is: $R^2=\frac{SSR}{SST}=1-\frac{SSE}{SST}$


**Interpretation**

- $R^2$ measures the proportion of the total variation in $Y$ that is explained by the regression model.
- For example: $R^2=0.80$ means that approximately $80\%$ of the sample variation in $Y$ is explained by the model.


### Matrix Form of the Variance of OLS Coefficients

- For the multiple regression model: $\mathbf{Y}=\mathbf{X}\boldsymbol{\beta}+\boldsymbol{\epsilon}$
- the OLS estimator is: $\hat{\boldsymbol{\beta}}=(\mathbf{X}^{T}\mathbf{X})^{-1}\mathbf{X}^{T}\mathbf{Y}$
- Under the classical homoscedasticity assumption:$\operatorname{Var}(\boldsymbol{\epsilon}\mid X)=\sigma^2I$
- the variance-covariance matrix of the OLS estimator is: $ \operatorname{Var}(\hat{\boldsymbol{\beta}}\mid X) =\sigma^2(\mathbf{X}^{T}\mathbf{X})^{-1}$
- Since $\sigma^2$ is unknown, we estimate it using: $\hat{\sigma}^2=MSE=\frac{SSE}{n-k-1}$ Therefore: $\widehat{\operatorname{Var}}(\hat{\boldsymbol{\beta}})=\hat{\sigma}^2(\mathbf{X}^{T}\mathbf{X})^{-1}$
- The standard error of the $j$-th coefficient is:$SE(\hat{\beta}_j)=\sqrt{\left[\hat{\sigma}^2(\mathbf{X}^{T}\mathbf{X})^{-1}\right]_{jj}}$

### Standard Error of Regression Coefficients

(1) **standard error of a coefficient** 
- it measures the uncertainty or sampling variability of its estimate. For example: $\hat{\beta}_1$ is our estimate of the true coefficient: $\beta_1$
- The standard error tells us how much $\hat{\beta}_1$ would tend to vary if we repeatedly collected new samples and estimated the regression each time.

Therefore:

- small standard error $\rightarrow$ precise estimate
- large standard error $\rightarrow$ uncertain estimate

(2) **$t$-Statistic**

- To test whether a coefficient is statistically significant: $t=\frac{\hat{\beta}_j-\beta_{j,0}}{SE(\hat{\beta}_j)}$
- For the common test:$ H_0:\beta_j=0$, the statistic becomes: $t=\frac{\hat{\beta}_j}{SE(\hat{\beta}_j)}$
- If the standard error is small relative to the estimated coefficient: $ SE(\hat{\beta}_j)\downarrow \Rightarrow |t|\uparrow$ and the coefficient is more likely to be statistically significant.
-  For: $Y_i=\beta_0+\beta_1X_i+\epsilon_i$, the standard error of the estimated slope is:

$$SE(\hat{\beta}_1)=\sqrt{\frac{\hat{\sigma}^2}{\displaystyle\sum_{i=1}^{n}(X_i-\bar X)^2}}$$

- Since:$\hat{\sigma}^2=\frac{SSE}{n-k-1}$
- for simple linear regression, $k=1$, so: $\hat{\sigma}^2=\frac{SSE}{n-2}$
- Therefore:$$SE(\hat{\beta}_1)=\sqrt{\frac{SSE}{(n-2)\displaystyle\sum_{i=1}^{n}(X_i-\bar X)^2}}$$


(3) **Understanding $\sum(X_i-\bar X)^2$**

- The quantity $\sum_{i=1}^{n}(X_i-\bar X)^2$ measures how much variation or information we have in the explanatory variable $X$.
- If the $X$ values are widely spread: $\sum(X_i-\bar X)^2\uparrow$ then: $SE(\hat{\beta}_1)\downarrow$

Therefore, more variation in $X$ gives us more information for estimating the slope.

Conceptually:

$$SE(\hat{\beta}_1)\sim\sqrt{\frac{\text{Noise}}{\text{Information in }X}}$$



(4) **When Does the Standard Error Increase?**

- The standard error of a coefficient generally increases when:

(a) The model is noisy
$SSE\uparrow
\Rightarrow
\hat{\sigma}^2\uparrow
\Rightarrow
SE(\hat{\beta}_1)\uparrow$

(b) Sample size is small- A smaller sample provides less information and generally produces less precise estimates.

(c) There is little variation in $X$
$\sum(X_i-\bar X)^2\downarrow
\Rightarrow
SE(\hat{\beta}_1)\uparrow$

Therefore:

$\text{More noise + less data + less variation in }X
\rightarrow
\text{larger standard error}$

Therefore:

$SSE\uparrow
\Rightarrow
MSE\uparrow
\Rightarrow
\hat{\sigma}^2\uparrow$

A larger unexplained variation generally means a noisier regression model.

### Big Picture Connection

| Quantity | Formula | Meaning |
|---|---|---|
| $SSE$ | $\displaystyle\sum_i\hat{\epsilon}_i^2$ | Unexplained variation |
| $MSE$ | $\displaystyle\frac{SSE}{n-k-1}$ | Estimated error variance |
| $\hat{\sigma}^2$ | $MSE$ | Estimated noise level |
| $R^2$ | $\displaystyle1-\frac{SSE}{SST}$ | Percentage of variation explained |
| $F$-statistic | $\displaystyle\frac{MSR}{MSE}$ | Signal-to-noise measure |

## Autocorrelation/Serial Correlation test

- In practice, autocorrelation and serial correlation are often used interchangeably, but there is a subtle distinction.
- Autocorrelation specifically means: Correlation of a variable with its own past values.
- Serial correlation is a broader term: Correlation between observations across time/order (serially ordered data).

### Ljung-Box test

- The Ljung–Box test is a statistical test used to check for serial correlation (autocorrelation) in a time series — especially in model residuals.
- Instead of checking autocorrelation at one lag (like Durbin-Watson), Ljung–Box checks multiple lags together

It tests whether a group of autocorrelations is jointly zero.

Null hypothesis (H₀): No serial correlation up to lag h <br>
Alternative hypothesis (H₁): At least one autocorrelation ≠ 0

$$ Q = n(n+2)\sum_{k=1}^{h} \frac{\hat{\rho}_k^2}{n-k} $$

Where:

n = sample size <br>
h = number of lags tested <br>
${\rho}^k$ = sample autocorrelation at lag k

👉 Under H0,
Q∼χ2(h)

👉 In simple terms:
It measures the largest gap between two cumulative distribution curves.

If p-value ≤ α → ❌ Reject H0 <br>
If p-value > α → ✅ Fail to reject H0


In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

# residuals from model
result = acorr_ljungbox(residuals, lags=[10], return_df=True)

print(result)

### Fisher transform test

- The Fisher transform test for serial correlation is based on applying the Fisher Z-transformation to the sample autocorrelation coefficient so that it becomes approximately normally distributed.

- Hypothesis <br>
We want to test whether there is serial correlation at lag k:
Null hypothesis: H0:ρk=0 <br>
Alternative: H1:ρk=0 <br>
where ρk is the population autocorrelation at lag k, and r k is the sample autocorrelation.

- Fisher Z-Transformation <br>
The issue is that $r_k$ is not normally distributed, especially for small samples. So we apply Fisher transform:
$$Z_k = \frac{1}{2} \ln \left( \frac{1 + r_k}{1 - r_k} \right) $$
This makes it approximately normal.

- Distribution of Transformed Statistic <br>
Under H0(no serial correlation):
$$ Z_k \sim N\left(0, \frac{1}{n-k-3}\right) $$

So the standardized test statistic is:
$$ T = \frac{Z_k}{\sqrt{1/(n-k-3)}} $$

which follows approximately:
$$T \sim N(0,1) $$

- Decision Rule <br>
Reject H0 if: $|T| > z_{\alpha/2}$
OR for a two tailed test $p\text{-value} = 2\left(1-\Phi(|T|)\right)$ <br>
If p-value < α → ❌ Reject H0 

In [ ]:
# the Fisher transform test for autocorrelation is NOT directly available in the statsmodels library.

import numpy as np
from scipy.stats import norm

def fisher_autocorr_test(series, lag=1, alpha=0.05):
    """
    Fisher transform test for autocorrelation at given lag
    
    Parameters:
        series : array-like time series
        lag    : lag at which autocorrelation is tested
        alpha  : significance level
    
    Returns:
        dict with r_k, test_stat, p_value, decision
    """
    
    series = np.asarray(series)
    n = len(series)
    
    # Step 1: compute autocorrelation at lag k
    x = series[:-lag]
    y = series[lag:]
    r_k = np.corrcoef(x, y)[0, 1]
    
    # Step 2: Fisher transform
    z = 0.5 * np.log((1 + r_k) / (1 - r_k))
    
    # Step 3: standard error
    se = np.sqrt(1 / (n - lag - 3))
    
    # Step 4: test statistic
    T = z / se
    
    # Step 5: p-value (two-tailed)
    p_value = 2 * (1 - norm.cdf(abs(T)))
    
    # Step 6: decision
    reject = p_value < alpha
    
    return {
        "autocorrelation (r_k)": r_k,
        "test_statistic": T,
        "p_value": p_value,
        "reject_H0": reject
    }

### Autocorrelation and Partial Autocorrelation Plots

An autocorrelation plot (also known as a [Correlogram](https://en.wikipedia.org/wiki/Correlogram) ) shows the correlation of the series with itself, lagged by x time units. So the y axis is the correlation and the x axis is the number of time units of lag.

So imagine taking your time series of length T, copying it, and deleting the first observation of copy #1 and the last observation of copy #2. Now you have two series of length T−1 for which you calculate a correlation coefficient. This is the value of of the vertical axis at x=1x=1 in your plots. It represents the correlation of the series lagged by one time unit. You go on and do this for all possible time lags x and this defines the plot.

You will run these plots on your differenced/stationary data. There is a lot of great information for identifying and interpreting ACF and PACF [here](http://people.duke.edu/~rnau/arimrule.htm) and [here](https://people.duke.edu/~rnau/411arim3.htm).

https://towardsdatascience.com/significance-of-acf-and-pacf-plots-in-time-series-analysis-2fa11a5d10a8


The actual interpretation and how it relates to ARIMA models can get a bit complicated, but there are some basic common methods we can use for the ARIMA model. Our main priority here is to try to figure out whether we will use the AR or MA components for the ARIMA model (or both!) as well as how many lags we should use. In general you would use either AR or MA, using both is less common.

* If the autocorrelation plot shows positive autocorrelation at the first lag (lag-1), then it suggests to use the AR terms in relation to the lag

* If the autocorrelation plot shows negative autocorrelation at the first lag, then it suggests using MA terms.

In general, a partial correlation is a conditional correlation.

It is the correlation between two variables under the assumption that we know and take into account the values of some other set of variables.

For instance, consider a regression context in which y = response variable and x1, x2, and x3 are predictor variables.  The partial correlation between y and x3 is the correlation between the variables determined taking into account how both y and x3 are related to x1 and x2.

Formally, this is relationship is defined as:

$$\frac{\text{Covariance}(y, x_3|x_1, x_2)}{\sqrt{\text{Variance}(y|x_1, x_2)\text{Variance}(x_3| x_1, x_2)}}$$

Check out this [link](http://www.itl.nist.gov/div898/handbook/pmc/section4/pmc4463.htm) for full details on this.

* Typically a sharp drop after lag "k" suggests an AR-k model should be used. If there is a gradual decline, it suggests an MA model.

* Identification of an AR model is often best done with the PACF.
    * For an AR model, the theoretical PACF “shuts off” past the order of the model.  The phrase “shuts off” means that in theory the partial autocorrelations are equal to 0 beyond that point.  Put another way, the number of non-zero partial autocorrelations gives the order of the AR model.  By the “order of the model” we mean the most extreme lag of x that is used as a predictor.
    
    
* Identification of an MA model is often best done with the ACF rather than the PACF.
    * For an MA model, the theoretical PACF does not shut off, but instead tapers toward 0 in some manner.  A clearer pattern for an MA model is in the ACF.  The ACF will have non-zero autocorrelations only at lags involved in the model.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plot_acf(df['seasonal first diff'].dropna())                         # plot_acf()
plot_pacf(df['seasonal first diff'].dropna())                        # plot_pacf()

### Pearson's $\chi^2$ Test

This test is used to compare distributions

Let's revisit Tommy, the cheating bastard he is. Last time we tried to answer the question *"Is Tommy rolling too many sixes?"*.

Now, time to ask a different question. What if we're not worried about just the number of sixes, but what to ask *"Is the distribution of rolls we get consistent with a fair die?"*

The Pearson's $\chi^2$ test for rolling a die gives

$$ \chi^2 = \sum_{i=1}^{6} \frac{(C_i - E_i)^2}{E_i} $$

here, for each outcome $C_i$ mean count of outcome i i.e. 1,2,3,4,5 and 6 and $E_i$ mean expected outcome
But what does this $\chi^2$ value mean? We can convert it to a probability given the $\chi^2$ distribution, with 5 degrees of freedom (six sides - 1)

In [ ]:
data = np.loadtxt(r"C:\Users\Admin\Python Statistical Analysis\Hypothesis Testing\Distributions\loaded_500.txt")

expected = data.size/6
values, counts = np.unique(data,return_counts=True)
chi2_value = np.sum((counts - expected)**2/expected)

test_statistics = chi2_value

# to create hypothetical distribution

hypothetical_dist = np.linspace(0,15,500)
pdf_chi2 = scs.chi2.pdf(hypothetical_dist,df=5)                  # scs.chi2.pdf()
p_vaue = scs.chi2.sf(test_statistics, df=5)                      # scs.chi2.sf()

plt.figure(figsize =(12,8))
plt.plot(hypothetical_dist,pdf_chi2)
plt.axvline(test_statistics, ls = "--", c = "red")
plt.fill_between(x = hypothetical_dist,y1 = pdf_chi2,where = ( hypothetical_dist > chi2_value), color = "green" )
plt.show()

## Goodness of fit test - AD, CVM, KS, Shapiro-Wilk,


\begin{array}{|l|c|l|}
\hline
\textbf{Test} & \textbf{Works for any distribution?} & \textbf{Best use} \\
\hline
\text{Shapiro\textendash Wilk} & \text{No} & \text{Normality (most powerful)} \\
\hline
\text{Kolmogorov\textendash Smirnov (KS)} & \text{Yes} & \text{General comparison} \\
\hline
\text{Anderson\textendash Darling (AD)} & \text{Yes} & \text{Tail-sensitive testing} \\
\hline
\text{Cram\'er\textendash von Mises (CVM)} & \text{Yes} & \text{Overall fit} \\
\hline
\end{array}


\begin{array}{|l|c|l|}
\hline
\textbf{Test} & \textbf{Two-sample version?} & \textbf{Common usage} \\
\hline
\text{Kolmogorov\textendash Smirnov (KS)} & \text{Yes} & \text{Empirical vs empirical} \\
\hline
\text{Cram\'er\textendash von Mises (CVM)} & \text{Yes} & \text{Less common} \\
\hline
\text{Anderson\textendash Darling (AD)} & \text{Yes} & \text{Rare in practice} \\
\hline
\end{array}


👉 CVM, Anderson–Darling (AD), and Kolmogorov–Smirnov (KS) are all goodness-of-fit tests, but:

❗ They are primarily designed to compare an empirical distribution with a theoretical distribution, not strictly “two empirical distributions.”

However:

KS → has a well-known two-sample version (empirical vs empirical) <br>
CVM & AD → can also be extended to two-sample cases, but are less commonly used that way

1. What Each Test Actually Does<br>

✅ Kolmogorov–Smirnov test
Measures maximum distance between CDFs
Works for:
One-sample (empirical vs theoretical)
Two-sample (empirical vs empirical)

👉 Focus: largest deviation anywhere

✅ Cramér–von Mises test
Measures integrated squared difference between CDFs

👉 Focus: overall deviation across all points

✅ Anderson–Darling test
Weighted version of CVM
Gives more importance to tails

👉 Focus: tail differences

### Kolmogorov Smirnov  Test

- these tests are used to compare distributions, its non-parametric test means no assumption about normality
- basically, we take distance between both empirical distributions in CDF curve. 
- H0  Null hypothesis - two distribution are drwan from same population, H1: two distributions are not not from simialr population
- test statistics = max distance of two cdf distribution and compare it with critical value, if D > critical value then reject null hypothese
- p value = probability of getting this max distance, had the two empirical distribution drawn from same underlying population
- it is meant for continuous distributions,In fact, the reason we are using 10 dice added together is to make our discrete distribution continuous enough.

3. Hypotheses<br>
One-sample:H0: Data follows the specified distribution
H1: Data does NOT follow that distribution
<br>
Two-sample:H0: Both samples come from the same distribution
H1: They come from different distributions

$$D = \sup_x \left|F_n(x) - F(x)\right|$$

Where:

$F_n(x)$ = empirical CDF (from data) <br>
$F(x)$ = theoretical CDF (or another sample CDF) <br>
$D$ = maximum absolute difference

👉 In simple terms:
It measures the largest gap between two cumulative distribution curves.

If p-value < α → ❌ Reject H0 <br>
If p-value > α → ✅ Fail to reject H0
	​


In [ ]:
rolls_obs = np.loadtxt(r"C:\Users\Admin\Python Statistical Analysis\Hypothesis Testing\KS_and_AD\loaded_many_100.txt")
unique_1,count_obs_1 = np.unique(rolls_obs, return_counts=True)
count_obs_1 = count_obs_1/np.sum(count_obs_1)                               # to normalize i.e. density = True
cdf_obs_1 = np.cumsum(count_obs_1)                                          # np.cumsum()   

rolls_fair = np.random.randint(low=1, high=7,size =(50000,10)).sum(axis=1)
unique_2,count_obs_2 = np.unique(rolls_fair, return_counts=True)          
count_obs_2 = count_obs_2/np.sum(count_obs_2)                               # to normalize i.e. density = True
cdf_obs_2 = np.cumsum(count_obs_2)                                          # np.cumsum()

plt.plot(unique_1,cdf_obs_1, label = "data")
plt.plot(unique_2,cdf_obs_2, label = "fair")
plt.legend()
plt.show()

test_statistics, p_value = scs.ks_2samp(rolls_obs,rolls_fair)               # scs.ks_2samps()

# Generate sample data
data = np.random.normal(0, 1, 100)

# One-sample K-S test (test normality)
stat, p_value = scs.kstest(data, 'norm')                                   # scs.kstest()

print("K-S Statistic:", stat)
print("p-value:", p_value)

### CVM Test

- The Cramér–von Mises (CvM) test is a goodness-of-fit test used to check whether a sample comes from a specified distribution (like normal, exponential, etc.).
- Idea behind the CvM Test, It measures the distance between: Empirical CDF (based on your sample), and Theoretical CDF (expected distribution)
- Unlike KS test (which looks at max difference), CvM considers overall squared differences across all points.
- The Cramér–von Mises (CvM) test statistic $W^2$ does not follow a standard named distribution like normal, t, or chi-square.Cramér–von Mises distribution (a special, non-standard distribution)

$$W^2 = \frac{1}{12n} + \sum_{i=1}^{n} \left( F(x_{(i)}) - \frac{2i - 1}{2n} \right)^2$$

Where:

n = sample size <br>
$X_i$= ordered sample values  <br>
F(x) = theoretical CDF 

H₀: Data follows the specified distribution
H₁: Data does NOT follow that distribution

Decision Rules, 2 ways
1. Small $W^2$→ good fit and Large $W^2$ → poor fit
2. p>α → Do NOT reject H₀ (good fit) and p≤α → Reject H₀ (bad fit)

In [ ]:
# Example data
data = [2.1, 2.5, 3.0, 3.2, 3.8]

# CvM test for normal distribution
result = scs.cramervonmises(data, 'norm')

print("Statistic:", result.statistic)
print("p-value:", result.pvalue)

### Anderson-Darling Test

- The Anderson–Darling test (AD test) is a statistical test used to check whether a sample comes from a specific distribution
- An improvement on the KS test that gives more weight to the tails of the distribution (whilst the K-S test gives more weight in the center). The default test requires testing against analytic distributions (like the KS test), however you can also test two observed distributions.
- The AD test statistic $W^2$ does not follow a standard named distribution like normal, t, or chi-square. AD distribution (a special, non-standard distribution)

$$A^2 = -n - \frac{1}{n} \sum_{i=1}^{n} (2i - 1)
\left[
\ln F(X_{(i)}) +
\ln\left(1 - F(X_{(n+1-i)})\right)
\right]$$

Where:

n = sample size <br>
$X_i$= ordered sample values  <br>
F(x) = theoretical CDF (e.g., normal CDF) <br>

Null Hypothesis (H₀): The sample comes from the specified distribution

Decision Rule
Two ways:

1. Using critical values
Compare $A^2$ with table value If: $ A^2 > critical value⇒Reject H_0 $ <br>

2. Using p-value
If: $p-value < α ⇒ Reject H0$
	​


In [ ]:
# Sample data
data = np.random.normal(loc=0, scale=1, size=100)

# Anderson-Darling test for normality
result = scs.anderson(data, dist='norm')                       # scs.anderson()

print("Test Statistic:", result.statistic)
print("Critical Values:", result.critical_values)
print("Significance Levels:", result.significance_level)


### Shapiro-Wilk

- The Shapiro–Wilk test is a statistical test used to check whether a dataset follows a normal distribution.
- The Shapiro–Wilk test statistic (W) does NOT follow a standard named distribution (like t, χ², or F).

Tests the null hypothesis: H₀: Data is normally distributed <br>
Alternative hypothesis: H₁: Data is NOT normally distributed

$$W = \frac{\left( \sum_{i=1}^{n} a_i x_{(i)} \right)^2} {\sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Where:

$x_{(i)}$ : ordered sample values (sorted) <br>
$a_i$ : constants based on expected values of normal order statistics <br>
: sample mean

W close to 1 → data is likely normal W far from 1 → data deviates from normality <br>
p-value:
p > α (e.g., 0.05) → fail to reject H₀ → data is normal
p ≤ α → reject H₀ → data is not normal

In [ ]:

data = [12, 15, 14, 10, 13, 12, 11]

stat, p_value = scs.shapiro(data)                              # scs.shapiro()

print("W Statistic:", stat)
print("p-value:", p_value)

if p_value > 0.05:
    print("Data looks normal (fail to reject H0)")
else:
    print("Data is not normal (reject H0)")

## VaR backtesting

- These are the standard backtesting tests used for evaluating Value-at-Risk (VaR) models.
- Kupiec Test (Proportion of Failures Test), Christoffersen Independence Test, Christoffersen Conditional Coverage Test.


### Kupiec Test (Proportion of Failures Test)

Also called:
- POF Test
- Unconditional Coverage Test

It checks whether the observed exception frequency matches the expected VaR confidence level.

Example: <br>
99% VaR, Expected breach probability = 1%, Over 1000 days → expected ≈ 10 breaches
If actual breaches are very different from expected, the VaR model may be incorrect, it checks for both overestimation of breaches and underestimation of breaches unlike binomial model where we only check of overestimation of breaches

$T$ = total observations <br>
$x$ = number of VaR breaches <br>
$p$ = expected breach probability 


Then:
H_0: $\pi = p $ <br>
H_1: $\pi \neq p$<br>
where:
$\pi = \frac{x}{T}$

- Likelihood Ratio Statistic
$$LR_{POF} =-2 \ln\left[\frac{(1-p)^{T-x} p^x}{(1-\pi)^{T-x} \pi^x}\right] $$
$$LR_{POF} \sim \chi^2(1)$$

Under the null hypothesis, the Kupiec test statistic approximately follows a Chi-square distribution with 1 degree of freedom.
- Typical procedure:<br>
Compute the test statistic LR POF and Compute the critical value from $\chi^2(1)$ and Reject or fail to reject the VaR model <br>

At 5% significance level:
Critical value is approximately:<br>
$\chi^2_{0.95}(1)≈3.84 $ <br>
Decision rule: If $LR_{POF}$ >3.84 → reject the VaR model <br>
Otherwise:$LR_{POF}$ ≤ 3.84 → do not reject the VaR model


In [ ]:
import numpy as np
from scipy.stats import chi2

def kupiec_test(exceptions, alpha=0.99):
    """
    exceptions:
        1 if VaR breach occurred
        0 otherwise

    alpha:
        VaR confidence level
    """

    exceptions = np.array(exceptions)

    T = len(exceptions)
    x = exceptions.sum()

    p = 1 - alpha
    pi = x / T

    # avoid log(0)
    if pi == 0 or pi == 1:
        return {
            "LR": np.inf,
            "p_value": 0.0,
            "reject": True
        }

    LR = -2 * np.log(
        ((1 - p)**(T - x) * (p**x)) /
        ((1 - pi)**(T - x) * (pi**x))
    )

    p_value = 1 - chi2.cdf(LR, df=1)

    return {
        "LR": LR,
        "p_value": p_value,
        "reject_5pct": p_value < 0.05,
        "exceptions": int(x),
        "expected": T * p
    }

### Christoffersen Independence Test

- Kupiec only checks number of breaches.Christoffersen Independence Test checks whether breaches are independent over time.
- A good VaR model should not produce clustered breaches.
- Example of clustering: 0 0 0 1 1 1 0 0 : This indicates volatility regime changes not captured properly. <br>
$N_{00}$: $0 \rightarrow 0$ transitions <br>
$N_{01}$: $0 \rightarrow 1$ transitions <br>
$N_{10}$: $1 \rightarrow 0$ transitions <br>
$N_{11}$: $1 \rightarrow 1$ transitions <br>

$$ \pi_{01}=\frac{N_{01}}{N_{00} + N_{01}}$$
$$ \pi_{11}=\frac{N_{11}}{N_{10} + N_{11}}$$
$$ \pi=\frac{N_{01}+N_{11}}{N_{00}+N_{01}+N_{10}+N_{11}}$$

Under independence: $$ \pi_{01} = \pi_{11} $$

- Test Statistic <br>
$$LR_{IND} =-2 \ln\left(\frac{L_0}{L_1}\right)$$
where,<br>
Restricted likelihood:
$$L_0=(1-\pi)^{N_{00}+N_{10}}\pi^{N_{01}+N_{11}}$$
Unrestricted likelihood:
$$L_1=(1-\pi_{01})^{N_{00}}\pi_{01}^{N_{01}}(1-\pi_{11})^{N_{10}}\pi_{11}^{N_{11}}$$

Under H0 :
$$LR_{IND} \sim \chi^2(1)$$

- Under the null hypothesis, the Kupiec Ind test statistic approximately follows a Chi-square distribution with 1 degree of freedom.
- Typical procedure:<br>
Compute the test statistic LR IND and Compute the critical value from $\chi^2(1)$ and Reject or fail to reject the VaR model <br>

At 5% significance level:
Critical value is approximately:<br>
$\chi^2_{0.95}(1)≈3.84 $ <br>
Decision rule: If $LR_{IND}$ >3.84 → reject independence <br>
Otherwise:$LR_{IND}$ ≤ 3.84 → do not reject independence

In [ ]:
import numpy as np
from scipy.stats import chi2

def christoffersen_independence_test(exceptions):

    exceptions = np.array(exceptions)

    N00 = N01 = N10 = N11 = 0

    for i in range(1, len(exceptions)):
        prev_ = exceptions[i - 1]
        curr_ = exceptions[i]

        if prev_ == 0 and curr_ == 0:
            N00 += 1
        elif prev_ == 0 and curr_ == 1:
            N01 += 1
        elif prev_ == 1 and curr_ == 0:
            N10 += 1
        elif prev_ == 1 and curr_ == 1:
            N11 += 1

    pi01 = N01 / (N00 + N01) if (N00 + N01) > 0 else 0
    pi11 = N11 / (N10 + N11) if (N10 + N11) > 0 else 0

    pi = (N01 + N11) / (N00 + N01 + N10 + N11)

    def safe_log(x):
        return np.log(x) if x > 0 else -1e10

    L0 = (
        ((1 - pi) ** (N00 + N10)) *
        (pi ** (N01 + N11))
    )

    L1 = (
        ((1 - pi01) ** N00) *
        (pi01 ** N01) *
        ((1 - pi11) ** N10) *
        (pi11 ** N11)
    )

    LR = -2 * np.log(L0 / L1)

    p_value = 1 - chi2.cdf(LR, df=1)

    return {
        "LR": LR,
        "p_value": p_value,
        "reject_5pct": p_value < 0.05,
        "N00": N00,
        "N01": N01,
        "N10": N10,
        "N11": N11
    }

### Christoffersen Conditional Coverage Test

- This combines: Kupiec coverage test & Independence test So it checks: Correct breach frequency and No clustering
- Test Statistic
$$ {LR}_{CC} =  {LR}_{POF} + {LR}_{IND} $$
$$ LR_{CC} \sim \chi^2(2)$$

- Under the null hypothesis, the Conditional Coverage test statistic follows a Chi-square distribution with 2 degrees of freedom.

- Meaning of $\chi^2(2)$ <br>
means a Chi-square distribution with 2 degrees of freedom.
The “2” comes from testing two restrictions jointly: Correct exception probability & Independence of exceptions Hence:1+1=2 degrees of freedom.

- At 5% significance level:<br>
Critical value is approximately:<br>
$\chi^2_{0.95}(2)≈5.99 $ <br>
Decision rule: If $LR_{CC}$ >5.99 → reject the VaR model <br>
Otherwise:$LR_{CC}$ ≤ 5.99 → do not reject the VaR model


In [ ]:
from scipy.stats import chi2

def conditional_coverage_test(exceptions, alpha=0.99):

    kupiec = kupiec_test(exceptions, alpha)
    ind = christoffersen_independence_test(exceptions)

    LRcc = kupiec["LR"] + ind["LR"]

    p_value = 1 - chi2.cdf(LRcc, df=2)

    return {
        "LR_CC": LRcc,
        "p_value": p_value,
        "reject_5pct": p_value < 0.05,
        "kupiec_LR": kupiec["LR"],
        "independence_LR": ind["LR"]
    }

# Part B : From Panda Course

## stastical data visualtization with seaborn

In [ ]:
import seaborn as sns

titanic = pd.read_csv('C:\\Users\\Admin\\Pandas Course\\Course_Materials_Part2\\Video_Lecture_NBs\\titanic.csv')
# to adjust height/width of graph, use height/ratio attributes of sns func - plt.figure() only for those where these are missing


#--------------------------------------------------------------------------------------------------------------
# categorical plots

# countplot()
# hue takes categorical variable
plt.figure(figsize =(12,8))
sns.set(font_scale =2,palette = 'viridis')                                   # sns.set()
sns.countplot(data = titanic,x = 'sex', hue = 'pclass')                      # sns.countplot() 
plt.show()


# striplplot()
# Draw a scatterplot where one variable is categorical
# dodge =True means separate strips
plt.figure(figsize =(12,8))
sns.set(font_scale =1.2)
sns.stripplot(data = titanic, x = 'sex',y='age', jitter = True, hue = 'pclass',dodge =True)        # sns.stripplot()
plt.show()


# swarmplot() - this shows frequency distribution also
plt.figure(figsize =(12,8))
sns.set(font_scale =1.2)
sns.swarmplot(data = titanic, x = 'sex',y='age', hue = 'pclass',dodge =True)                       # sns.swarmplot()
plt.show()


# violinplot() - better presentation of frequency distribution compared to swarmplot()
plt.figure(figsize =(12,8))
sns.set(font_scale =1.2)
sns.violinplot(data = titanic, x = 'sex',y='age', hue = 'pclass',dodge =True)                      # sns.violinplot()
sns.swarmplot(data = titanic, x = 'sex',y='age', hue = 'pclass',dodge =True, color = 'black')      # sns.swarmplot()
plt.show()


# barplot() 
# this shows mean age along Y axis and confidence interval(ci = 95% probability that mean will lie in that range)
plt.figure(figsize =(12,8))
sns.set(font_scale =1.2)
sns.barplot(data = titanic, x = 'sex',y='age', hue = 'pclass',dodge =True)                         # sns.barplot()
plt.show()


# pointplot() 
# this shows mean age along Y axis and confidence interval(ci = 95% probability that mean will lie in that range)
plt.figure(figsize =(12,8))
sns.set(font_scale =1.2)
sns.pointplot(data = titanic, x = 'sex',y='age', hue = 'pclass',dodge =True)                       # sns.pointplot()
plt.show()

#--------------------------------------------------------------------------------------------------------------
# joint plots
# Draw a plot of two variables with bivariate and univariate graphs(frequency distribution)
sns.set(font_scale =1.2)
#sns.jointplot(data = titanic, x = 'age', y = 'fare',kind = 'scatter')                      # sns.jointplot()
sns.jointplot(data = titanic, x = 'age', y = 'fare',kind = 'reg', ratio=5, height=8)        # reg - regression line
#sns.jointplot(data = titanic, x = 'age', y = 'fare',kind = 'kde')                          # kde - kernel density estimator
plt.show()

#--------------------------------------------------------------------------------------------------------------
# regression plots
sns.set(font_scale =1.2)
#sns.lmplot(data = titanic,x = 'age', y='fare',hue = 'sex',height =8, aspect=1)             # sns.lmplot()
sns.lmplot(data = titanic,x = 'age', y='fare',col = 'sex',height =8, aspect=1)              # col will create new subplots          
#sns.lmplot(data = titanic,x = 'age', y='fare',col = 'sex',height =8, aspect=1)             # rowl will create new subplots
sns.lmplot(data = titanic,x = 'age', y='survived',height =8, aspect=1,logistics = True)     # logistics regression - requires statsmodels package
plt.show()

#--------------------------------------------------------------------------------------------------------------
# matrixplots/heatmaps
# fmt means format for string representation for annotations
plt.figure(figsize =(12,8))
sns.set(font_scale =1.2)
sns.heatmap(titanic.corr(),annot= True,cmap = 'Reds', fmt ='0.2f',cbar = True)               # sns.heatmap()
# if you need to add x and y lables, df with without headers(only values)
#sns.heatmap(df,annot= True,cmap = 'Reds', fmt ='0.2f',cbar = True,xticklabels=tickers, yticklabels=tickers)
plt.show()



## statistics using scipy.stats

In [ ]:
# panda series and numpy array are similar so these can be used interchangeably for methods of scipy, panda, numpy....

np.set_printoptions(precision=4,suppress=True)                  # np.set_printoptions(), precision = # of decimals,suppress = True means no scientific

path1 = 'C:\\Users\\pgupt145\\Panda course\\Course_Materials_Statistics\\Video_Lectures_NBs\\SP500_pop.csv'
pop = np.loadtxt(path1, delimiter=',',usecols=1)*100            # np.loadtxt()

path2 = 'C:\\Users\\pgupt145\\Panda course\\Course_Materials_Statistics\\Video_Lectures_NBs\\sample.csv'
sample = np.loadtxt(path2, delimiter=',',usecols=1) *100 

np.isin(element=sample, test_elements=pop)                      # np.isin()
perc = [1.0,5.0,10.0]
my_array = np.random.randint(-1000,1000,size =200)

np.percentile(my_array,perc)                                    # np.percentile()

np.mean(pop)                                                    # np.mean()
np.median(pop)                                                  # np.median()
np.var(pop, ddof = 1)                                           # np.var() - variance, ddof = delta degree of freedom
np.std(pop, ddof = 1)                                           # np.std() - standard deviation
np.cov(a,b)                                                     # np.cov() - covariance matrix
np.corrcoef(a,b)                                                # np.corrcoef() - correlaion matrix

np.random.seed(123)                                             # np.random.seed() - to produce reproducibility
np.random.randint(low =1, high =11, size =10)                   # np.random.randint() - with replacement
np.random.random(10)                                            # np.random.random() - floats
np.random.random(10)*(100-50) + 50                              # np.random.random() - floats from [50,100)

np.random.uniform(low =1, high=11, size =10)                    # np.random.uniform() - to draw random values from uniform distribution
np.random.normal(size =10,loc =100,scale =5)                    # np.random.normal() - to draw random values from normal distribution

#--------------------------------------------------------------------------------------------------------------
# A central moment is a moment of a probability distribution  of a random variable  about the random variable’s mean
# First moment – mean(mu),Second moment – variance/standard deviation,Third moment – skewness,Fourth moment – kurtosis
# random numner generation function from np package and pdf,cdf and ppf(percentile point function) fom scs

path = 'C:\\Users\\pgupt145\\Panda course\\Coumergerse_Materials_Statistics\\Video_Lectures_NBs\\SP500_pop.csv'
pop = np.loadtxt(path, usecols=1, delimiter=',')

scs.describe(pop)                                               # scs.describe()
scs.skew(pop)                                                   # scs.skew() - to measure skewness
scs.kurtosis(pop,fisher = True)                                 # scs.kurtosis() - to measure fisher/pearson kurtosis
scs.zscore(pop)                                                 # scs.zscore() - zscore for the series

x1 = np.linspace(90,110,100)                                     
y1= scs.norm.pdf(loc=100, scale =5,x = x )                      # scs.norm.pdf() - probablity density function
x2 = np.linspace(90,110,100)                                     
y2 = scs.norm.cdf(loc=100, scale =5,x = x)                      # scs.norm.cdf() - cumulative density function

y3 = np.random.random(size =100) 
y3.sort()
x3 = scs.norm.ppf(loc=100, scale =5,y3)                         # scs.norm.ppf()- percentile point function , opposite of cdf()

#--------------------------------------------------------------------------------------------------------------
# to get 95th and 99th percentile (VaR calculation) and confidence level
scs.norm.ppf(0.95)
scs.norm.ppf(0.99)

conf = 0.9
scs.norm.interval(conf,loc = 0, scale =1)                        # scs.norm.interval() - confidence level 
left = scs.norm.interval(conf,loc = 0, scale =1)[0]
right = scs.norm.interval(conf,loc = 0, scale =1)[1]


plt.figure(figsize =(12,8))
plt.plot(x1,y1, color = 'red')
plt.xticks(ticks = np.arange(-4,4,1),labels=['-4','-3','-2','-1','0','1','2','3'])      
plt.plot(x2,y2, color = 'blue')
plt.plot(x3,y3)
plt.show()


scs.spearmanr(merged_df.Total,merged_df.participation, nan_policy = 'omit')  # scs.spearmanr() - hypothesis testing for spearman correlation

plt.figure(figsize =(20,8))
plt.grid()
plt.plot(x,y, color = 'red')
plt.fill_between(x,y,where = ((x>right)|(x<left)),color = 'blue', alpha =0.2)            # plt.fill_between()
plt.fill_between(x,y,where = ((x<right)&(x>left)),color = 'red', alpha =0.2)
plt.annotate('90%', xy=(-0.6,0.2), fontsize =35, color = 'green')                        # plt.annotate()
plt.annotate('5%', xy=(-2.1,0.02), fontsize =20, color = 'green')
plt.annotate('5%', xy=(1.9,0.02), fontsize =20, color = 'green')
plt.vlines(x = [-1,1],ymin=0,ymax = [scs.norm.pdf(-1), scs.norm.pdf(1)])                 # plt.vlines()
plt.title('Normal Probability Distribution', fontsize =18)
plt.xlabel('x values')
plt.ylabel('probability')
plt.xticks(ticks =np.arange(-4,5,1), labels=['-4','-3','-2','-1','0','1','2','3','4'])   # plt.xticks()
plt.show()

#--------------------------------------------------------------------------------------------------------------
# covariance and correlation with visualisation
path = 'C:\\Users\\Admin\\Python Pandas course\\Course_Materials_Statistics\\Video_Lectures_NBs\\movies_metadata.csv'

movie = pd.read_csv(path)
movie.release_date = pd.to_datetime(movie.release_date, errors='coerce')            # pd.to_datetime()                                               
movie = movie.set_index('release_date', drop= True)
movie.sort_index(inplace = True)
movie.budget = pd.to_numeric(movie.budget, errors='coerce')                         # pd.to_numeric()

df = movie.loc[:,['title','budget','revenue']]
df.iloc[:,-2:] = df.iloc[:,-2:]/1000000
df.dropna(axis =0, how='any', inplace=True)                                         # dropna() with how
df = df.loc[(df.revenue >0) & (df.budget >0),:]

df.cov()                                           # through dataframe
df.age.cov(df.revenue)                             # thourgh series

a = df.budget.to_numpy()                           # to_numpy()
b = df.revenue.to_numpy()
np.cov(a, b)                                       # np.cov() - thorugh numpy array

# parse_dates = to parse column in datetime format

df = pd.read_csv('C:\\Users\\pgupt145\\bud_vs_rev.csv',parse_dates= ['release_date'])  
df1 =df.copy()
df1.release_date = df.release_date.astype('str')
df1 = df1.loc[df1.release_date.str.contains('2016'),:]
df1.plot(kind = 'scatter',x = 'budget',y='revenue',figsize =(12,8))
plt.show()

#--------------------------------------------------------------------------------------------------------------
# linear regresion with scs
# slope coefficient = cov(xy)/var(x)
scs.linregress(x = df1.budget, y= df1.revenue)                          # scs.linregress()
np.polyfit(x = df1.budget, y = df1.revenue, deg=1)                      # we are passing panda series as ndarray


# Part C : Time Series Analysis

In [ ]:
from PIL import Image

import os
dir_path = r"C:\Admin\C Drive\Desktop\CQF Course\Python Labs"

### Timeseries data and SMA(Simple moving aveage)

In [ ]:
rom datetime import datetime

sample_datetime = [datetime(2016,1,1), datetime(2017,1,1)]
date_index = pd.DatetimeIndex(sample_datetime)                                # pd.DatetimeIndex()
date_index = pd.to_datetime(sample_datetime)                                  # pd.to_datetime() -preferable
date_index = pd.to_datetime(['01/01/16','02/01/16'], format= '%d/%m/%y')      # if not standard format YYYY-mm-dd then specify format in argument

data = np.random.random(4).reshape(2,2)
df = pd.DataFrame(data, columns = ['a','b'], index = date_index )

df.index.argmax()                                                             # argmax()- datetime index of max value
df.index.max()                                                                # max() - max value
df.index.argmin()                                                             # argmin() - datetime index of min value                                      
df.index.min()                                                                # min() - min value

#--------------------------------------------------------------------------------------------------------------
# time resampling and shift
path = 'C:\\Users\\Admin\\Python - Financial Analysis & Algo Trading\\Python-for-Finance-Repo-master\\05-Pandas-with-Time-Series\\time_data\\'
df = pd.read_csv(path + 'walmart_stock.csv')

df['Date'] = pd.to_datetime(df.Date)
df.set_index(keys ='Date',drop = True, inplace = True )

df.resample(rule = 'A').max()                                                                  # resample()

def first_day(df):
    return df[0]

df.resample(rule = 'A').apply(first_day)                                                       # apply()

df.resample(rule = 'A').apply(first_day).plot(y = 'Close',figsize =(12,8), kind = 'bar')       # resampling on dataframe
df.Close.resample(rule = 'A').apply(first_day).plot(figsize =(12,8), kind = 'bar')             # resampling on series
plt.show()

df.shift(periods =-1)                                             # shift()
df.tshift(freq = 'M')                                             # tshift() - same datetime index for frequency(copy same value to all datetime index)


#--------------------------------------------------------------------------------------------------------------
# rolling and expanding
# for rolling() function: min_periods  is Minimum number of observations in window required to have a value;
# otherwise, result is np.NaN.For a window that is specified by an integer, min_periods will default to window

df['15 days MA']  = df.Close.rolling(window=15).mean()            # rolling() - rolling window of 15 samples with current                                    
df['expanding MA'] = df.Close.expanding().mean()                  # expanding - window of all the previous points

df.loc[:,['Close','15 days MA','expanding MA']].plot(figsize=(16,6))
plt.show()

#--------------------------------------------------------------------------------------------------------------
# apply on rolling()
# rationale is you can use apply() function to anything which is collection object- series, df,  
# collection of dfs(groupby, resampling, rolling)
stock = yf.download(tickers = 'MSFT', start='2020-01-01', end = '2022-12-31', progress = False)
stock['Return'] = stock['Close'].pct_change(periods = 1)
df_stock = stock.loc[:,['Close','Return']]

def my_rolling_sharpe(y):
    return np.sqrt(22) * (y.mean() / y.std())

df_stock['Rolling Sharpe'] = df_stock['Return'].rolling(window = '30d').apply(my_rolling_sharpe) # rolling window of 30 calender days


#--------------------------------------------------------------------------------------------------------------
# bollinger bands

# close 20 days MA
df['20 days MA']  = df.Close.rolling(window=20).mean()

# upper band = 20MA + 2*std(20 days closing)
df['upper band']  = df['20 days MA'] + 2* df.Close...std()

# lower band = 20MA - 2*std(20 days closing)
df['lower band']  = df['20 days MA'] - 2* df.Close.rolling(window=20).std()

df.loc[:,['Close','20 days MA','upper band','lower band']].plot(figsize=(16,6))
plt.show()

### Time series analysis

In [ ]:
import statsmodels.api as sm

df = sm.datasets.macrodata.load_pandas().data                                         # sm.load_pandas().data - to load dataset
quarter_index = sm.tsa.datetools.dates_from_range(start = '1959Q1', end = '2009Q3')   # sm.tsa.datetools.dates_from_range()
df.index = quarter_index

### Trend and Cyclical

Below are the trends types

 - Trend can be upward, horizontal/stationary or downward
 - Seasonality means repeating trend i.e. ice-skating is high in winter months as compared to summer months.
 - Cyclical - Trends with no set repetition i.e Nifty with uptrend but no set repetition


The Hodrick-Prescott filter separates a time-series  y_t  into a trend  τ_t and a cyclical component  ζt

$y_t = \tau_t + \zeta_t$

The components are determined by minimizing the following quadratic loss function

$\min_{\\{ \tau_{t}\\} }\sum_{t}^{T}\zeta_{t}^{2}+\lambda\sum_{t=1}^{T}\left[\left(\tau_{t}-\tau_{t-1}\right)-\left(\tau_{t-1}-\tau_{t-2}\right)\right]^{2}$

In [ ]:
gdp_cycle, gdp_trend = sm.tsa.filters.hpfilter(x = df['realgdp'])     # sm.tsa.filters.hpfilter() - to get cyclical and trend
df['cycle'] = gdp_cycle  
df['trend'] = gdp_trend

df.loc[:,['realgdp','trend']].plot(figsize = (16,6))
plt.show()

### EWMA

EWMA will allow us to reduce the lag effect from SMA and it will put more weight on values that occured more recently (by applying more weight to the more recent values, thus the name). The amount of weight applied to the most recent values will depend on the actual parameters used in the EWMA and the number of periods given a window size.
[Full details on Mathematics behind this can be found here](http://pandas.pydata.org/pandas-docs/stable/computation.html#exponentially-weighted-windows)
or
https://towardsdatascience.com/time-series-from-scratch-exponentially-weighted-moving-averages-ewma-theory-and-implementation-607661d574fe

Here is the shorter version of the explanation behind EWMA.

The formula for EWMA is:

$ y_t =   \frac{\sum\limits_{i=0}^t w_i x_{t-i}}{\sum\limits_{i=0}^t w_i} $

Where x_t is the input value, w_i is the applied weight (Note how it can change from i=0 to t), and y_t is the output.

Now the question is, how to we define the weight term w_i ?

This depends on the adjust parameter you provide to the .ewm() method. where alpha is smoothing factor

When adjust is True (default), weighted averages are calculated using weights:

$y_t = \frac{x_t + (1 - \alpha)x_{t-1} + (1 - \alpha)^2 x_{t-2} + ... + (1 - \alpha)^t x_{0}}{1 + (1 - \alpha) + (1 - \alpha)^2 + ...+ (1 - \alpha)^t}$

When adjust=False is specified, moving averages are calculated as:

$\begin
{split}y_0 &= x_0 \\ 
y_t &= (1 - \alpha) y_{t-1} + \alpha x_t,\end{split}$

which is equivalent to using weights:

 \begin{split}w_i = \begin{cases}
    \alpha (1 - \alpha)^i & \text{if } i < t \\
    (1 - \alpha)^i        & \text{if } i = t.
\end{cases}\end{split}

When adjust=True we have y0=x0 and from the last representation above we have yt=αxt+(1−α)yt−1, therefore there is an assumption that x0 is not an ordinary value but rather an exponentially weighted moment of the infinite series up to that point.

One must have 0<α≤1, and while since version 0.18.0 it has been possible to pass α directly, it’s often easier to think about either the span, center of mass (com) or half-life of an EW moment:

\begin{split}\alpha =
 \begin{cases}
     \frac{2}{s + 1},               & \text{for span}\ s \geq 1\\
     \frac{1}{1 + c},               & \text{for center of mass}\ c \geq 0\\
     1 - \exp^{\frac{\log 0.5}{h}}, & \text{for half-life}\ h > 0
 \end{cases}\end{split}

  Span corresponds to what is commonly called an “N-day EW moving average”.
* Center of mass has a more physical interpretation and can be thought of in terms of span: c=(s−1)/2
* Half-life is the period of time for the exponential weight to reduce to one half.
* Alpha specifies the smoothing factor directly.

In [ ]:
path = 'C:\\Users\\Admin\\Python - Financial Analysis & Algo Trading\\Python-for-Finance-Repo-master\\08-Time-Series-Analysis\\'
airline = pd.read_csv(path + 'airline_passengers.csv')

airline['Month'] = pd.to_datetime(airline.Month, errors='coerce')
airline.set_index(['Month'], inplace = True)
airline.dropna(inplace = True)

airline['6M SMA'] = airline['Thousands of Passengers'].rolling(window = 6).mean()
airline['12M SMA'] = airline['Thousands of Passengers'].rolling(window = 12).mean()         # SMA - Simple moving average
airline['12M EWMA'] = airline['Thousands of Passengers'].ewm(span=12,adjust=True).mean()    # ewm() - Exponential moving average

### ETS (Error, Trend, Seasonality)

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

path = 'C:\\Users\\Admin\\Python - Financial Analysis & Algo Trading\\Python-for-Finance-Repo-master\\08-Time-Series-Analysis\\'
airline = pd.read_csv(path + 'airline_passengers.csv')

airline['Month'] = pd.to_datetime(airline.Month, errors='coerce')
airline.set_index(['Month'], inplace = True)
airline.dropna(inplace = True)

# ETS decomposition
# seasonal_decompose()
# model = multiplicative means trend is non-linear, addititive means trend is linear

result = seasonal_decompose(x = airline,model = 'multiplicative')   # seasonal_decompose()
result.plot()                                                       # shows all trend, seasonality and residual term graphs
result.trend                                                        # trend term
result.seasonal                                                     # seasonality term
result.resid                                                        # residual term

### Linear Regression

In [ ]:
# RSS = residual sum of squares
# SSE = sum of squares of error = RSS
# SSR = sum of squares of regression, some people call it sum of square of residuals , very confusing !! be careful
# SST = SSR + RSS/SSE

tickers = ['XOM', 'CVX','^GSPC']
prices = yf.download(tickers, start='2015-01-01', end='2022-12-31', progress = False)
raw_data = prices['Close']
stock_1 = tickers[0]
stock_2 = tickers[1]

# y = a + bX
X = sm.add_constant(data = raw_data[stock_1])      # sm.add_constant() - to add constant term to X(indepen variable)
y = raw_data[stock_2]
model = sm.OLS(endog = y , exog = X)               # sm.OLS()
results = model.fit()                              # fit()
a = results.params[0]                              # params()
b = results.params[1]             

results.summary()                                  # summary()
results.ssr                                        # ssr - sum of squares of  residuals and not sum of square of regression!
results.resid                                      # resid - residuals (y- y_hat), if squared then equals above calculation

# laTeX export
print(results.summary().as_latex())

### Cointegration test

result = coint(raw_data['XOM'],raw_data['CVX'])            # coint()
score = result[0]                                          # results[0] - cointegration statistics
p_value = result[1]                                        # results[1] - p value 

## Different Types of Models

### AR Model

In [ ]:
path = "C:\Admin\C Drive\Desktop\CQF Course\Python Labs\Self VAR\ice_cream.csv"
ice_cream = pd.read_csv(path, parse_dates=[0], index_col=0)
ice_cream = ice_cream.loc['2010-01-01':]

# firts peak is for 0th lag so not relevant
# anything under blue region is not statistically significant
# for AR model ACF is decaying which is the case here so this is definitely AR model
# plot_acf() and plot_pacf() are not final selection critria for lags selection, these are only starting points

plot_acf(x = ice_cream, lags = 100);                       # plot_acf()

# firts peak is for 0th lag so not relevant
# based on analysis, you can consider 1,2,9,10,11 lags for AR model as a starting point

plot_pacf(ice_cream, lags = 20);                           # plot_pacf()

# AR model
model = AutoReg(ice_cream,lags = 10)                       # AutoReg()
fitted_model = model.fit()                                 # fit()
fitted_model.resid                                         # resid
fitted_model.params                                        # params
fitted_model.summary()                                     # summary()
fitted_model.fittedvalues                                  # fittedvalues
fitted_model.aic                                           # aic
fitted_model.bic                                           # bic

# ARIMA model
model = ARIMA(ice_cream,order = (10,0,0))                  # ARIMA()
fitted_model = model.fit()                                 # fit()
fitted_model.resid                                         # resid
fitted_model.params                                        # params
fitted_model.summary()                                     # summary()
fitted_model.fittedvalues                                  # fittedvalues
fitted_model.aic                                           # aic
fitted_model.bic                                           # bic

### VAR Model(Vector Autoregressive Model)

In [ ]:
# Make sure time series used in VAR model are stationary

filepath = r"C:\Admin\C Drive\Desktop\CQF Course\Python Labs\Self VAR\file_gdp.txt"
macro_data = pd.read_csv(filepath, parse_dates=['date'], index_col='date')
macro_data.head()

# To perform ADF test - these variables are I(1) processes
adf_test = adfuller(macro_data['rgnp'].diff(periods = 1)[1:])
index = ['test statistics','p-value','# of lagged values','# of observations']
adf_output_1 = pd.Series(adf_test[0:4],index = index)
adf_output_1

adf_test = adfuller(macro_data['ulc'].diff(periods = 1)[1:])
index = ['test statistics','p-value','# of lagged values','# of observations']
adf_output_2 = pd.Series(adf_test[0:4],index = index)
adf_output_1, adf_output_2

#--------------------------------------------------------------------------------------------------------------
# To perform Granger Causality test
# if p value is less than 0.05 then we can say that second variable causes first variable
# four tests are performed, however results for all four very similar
# we checked all four cases(lag 1, lag 2 , lag 3 and lag4) imply- rgnp causes ulc 

print('rgnp causes ulc?\n')
print('------------------')
granger_1 = grangercausalitytests(x = macro_data[['ulc','rgnp']] , maxlag = 4 )                 # grangercausalitytests() 

print('\nulc causes rgnp?\n')
print('------------------')
granger_2 = grangercausalitytests(macro_data[['rgnp', 'ulc']], 4)

#--------------------------------------------------------------------------------------------------------------
# to fit VAR model

df = macro_data[['ulc','rgnp']]
train_df = df.iloc[:-12]
test_df = df.iloc[-12:]

# VAR Model to select optimal lags
# shows that lag 4 AIC-BIC values are lowest and hence 4 is optimal lag 
model = VAR(endog = train_df.diff(periods = 1)[1:])                                   # VAR()
sorted_order = model.select_order(maxlags = 20)                                       # select_order()
sorted_order.summary()                                                                # summary()

# here we are passing non-stationary varibles with enforce stationarity condition - not a good idea for only VAR model
# order means (p,q) - lag p AR and lag q MA
# this is ARMA model fitting (VARMAX)
var_model = VARMAX(endog = train_df,order =(4,0),enforce_stationarity=True)           # VARMAX()
fitted_model = var_model.fit(disp = False)                                            # fit()
fitted_model.resid                                                                    # resid
fitted_model.params                                                                   # params
fitted_model.summary()                                                                # summary() 
fitted_model.fittedvalues                                                             # fittedvalues
fitted_model.aic                                                                      # aic
fitted_model.bic                                                                      # bic

# VAR model
model = VAR(endog = train_df.diff(periods = 1)[1:])                                   # VAR() 
fitted_model = model.fit(maxlags=10,ic = 'aic')                                       # fit()
fitted_model.resid                                                                    # resid
fitted_model.params                                                                   # params
fitted_model.summary()                                                                # summary()  
fitted_model.fittedvalues                                                             # fittedvalues
fitted_model.aic                                                                      # aic
fitted_model.bic                                                                      # bic

### ARIMA(Autorgressive Integrated Moving Average)

for ARIMA model, time series has to be stationary
* p - number of lag observations included in the model
* d - number of times raw observations are differenced to make the time series stationary
* q - size of moving average window(order of moving)

* identification of AR model is often done with PACF
* identification of MA model is often done with ACF

Genral purpose of ARIMA model is the following
* Visualize the time series data
* make the time series data stationary
* plot the correlation and autocorrelation charts
* construct the ARIMA model
* use the model to make predictions

In [ ]:
path = 'C:\\Users\\Admin\\Python - Financial Analysis & Algo Trading\\Python-for-Finance-Repo-master\\08-Time-Series-Analysis\\'

df = pd.read_csv(path + 'monthly-milk-production-pounds-p.csv')
df.columns = ['Month','Milk prod']
df.drop(index = 168, inplace = True)
df['Month'] = pd.to_datetime(df.Month)
df.set_index('Month', drop = True, inplace = True)

# visualization of time series

df.rolling(window=12).mean().plot(label = '12 months MA')
df.plot()

decom = seasonal_decompose(df,model = 'additive')
fig = decom.plot()
fig.set_size_inches(15,8)                                         # set_size_inches()

## Testing for Stationarity 

- Stationarity is important because most classical time-series models assume that the statistical properties of the series remain stable over time.
- Without stationarity: model parameters become unreliable, forecasts become unstable,statistical inference becomes invalid.
- What Does Stationarity Mean?
- A stationary time series has: Constant mean, Constant variance, Constant autocorrelation structure
- Why It Matters  Models such as below assume the time series has a constant autocorrelation structure, which is why we test stationarity before applying these models. <br>
  (a) ARIMA, <br>
  (b) AR model <br>
  (c) MA model <br>
  (d) VaR models
  

### ADF (Augmented Dickey Fuller Test)

A stationary timeseries has constant mean, variance and constant correlation structure

For a stationary series:
$ \mathrm{Corr}(Y_t, Y_{t-k}) = \mathrm{Corr}(Y_{t+h}, Y_{t+h-k}) $

This means:

- The autocorrelation depends only on the lag 𝑘
- It does NOT depend on time 𝑡

These values remain the same throughout the entire series. So the structure of dependence between observations is constant.<br>

\begin{array}{|c|c|}
\hline
\text{Lag} & \text{Autocorrelation} \\
\hline
1 & 0.6 \\
\hline
2 & 0.3 \\
\hline
3 & 0.1 \\
\hline
\end{array}

In the basic Dickey–Fuller test, the model is:
$\Delta y_t = \alpha + \beta y_{t-1} + \epsilon_t$
But residuals may have autocorrelation, which invalidates the test.

When the ADF test runs, it estimates the regression:
$\Delta y_t = \alpha + \beta y_{t-1} + \sum_{i=1}^{p} \gamma_i \Delta y_{t-i} + \epsilon_t$
These additional terms capture serial correlation in the series.

The function automatically chooses the lag length using:

- Akaike Information Criterion
- Bayesian Information Criterion
- t-stat method

💡 Practical intuition

Too few lags → residual autocorrelation → invalid test

Too many lags → loss of power

So ADF tries to find an optimal lag length automatically.

<br><br>



----------------------------------------------------------------------------------------------------------------
We can use the Augmented [Dickey-Fuller](https://en.wikipedia.org/wiki/Augmented_Dickey%E2%80%93Fuller_test) [unit root test](https://en.wikipedia.org/wiki/Unit_root_test).

In the Augmented Dickey–Fuller test, “lags used” means: The number of lagged differences of the time series included in the ADF regression to remove autocorrelation in the residuals.

In statistics and econometrics, an augmented Dickey–Fuller test (ADF) tests the null hypothesis that a unit root is present in a time series sample. The alternative hypothesis is different depending on which version of the test is used, but is usually stationarity or trend-stationarity.

Basically, we are trying to whether to accept the Null Hypothesis 

**H0** (that the time series has a unit root, indicating it is non-stationary) or 

reject **H0** and go with the Alternative Hypothesis (that the time series has no unit root and is stationary).

We end up deciding this based on the p-value return.

* A small p-value (typically ≤ 0.05) indicates strong evidence against the null hypothesis, so you reject the null hypothesis.

* A large p-value (> 0.05) indicates weak evidence against the null hypothesis, so you fail to reject the null hypothesis.

Let's run the Augmented Dickey-Fuller test on our data:

In [ ]:
from statsmodels.tsa.stattools import adfuller

data = pd.read_csv("D:/Programming/PythonProjects/PracticeWork/EFFR_SOFR.csv")
data_effr = data['EFFR'].dropna()

adf_result = adfuller(data_effr)
print("ADF Statistic" , adf_result[0])
print("p-value"       , adf_result[1])
print("Lags Used"     , adf_result[2])
print("Critical Vlaues" , adf_result[4])

### KPSS test

The KPSS test is used to check whether a time series is stationary around a deterministic trend or level.

Null hypothesis (H₀): The series is stationary

Alternative hypothesis (H₁): The series is non-stationary (has a unit root)

This is opposite of the Augmented Dickey–Fuller test, where the null hypothesis is non-stationarity.

p-value	Conclusion

p > 0.05	Fail to reject H₀ → Series is stationary

p ≤ 0.05	Reject H₀ → Series is non-stationary

In [ ]:
from statsmodels.tsa.stattools import kpss

result = kpss(data_effr,regression = 'c')                                                      # kpss()
print('KPSS Statistic:', result[0])
print('p-value:', result[1])
print('Lags Used:', result[2])
print('Critical Values:', result[3])

### Example 1

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adfuller_test(ds):
    results = adfuller(ds)               # adfuller() - augmented dickey-fuller test
    
    labels =['test statistics','p-value','# of lagged values','# of observations']
    
    for label,result in zip(labels, results):
        print(label + ' : ' + str(result))

#--------------------------------------------------------------------------------------------------------------
# ADF on time series      
# time series is not stationary
adfuller_test(df['Milk prod'].dropna())         

#--------------------------------------------------------------------------------------------------------------
# first diff ADF
# time series is stationary
df['first diff'] = df.diff(periods=1) 
adfuller_test(df['first diff'].dropna())         

#--------------------------------------------------------------------------------------------------------------
# seasonal diff ADF
# time series is not stationary
df ['seasonal diff'] = df['Milk prod'].diff(periods =12)
adfuller_test(df['seasonal diff'].dropna())         

#--------------------------------------------------------------------------------------------------------------
# seasonal first diff ADF
# very low p value, idea scenario
df['seasonal first diff'] = df['first diff'].diff(periods =12)   
adfuller_test(df['seasonal first diff'].dropna())         


### Example 2

In [ ]:
# generate AR model series
def generate_ar_process(lags, coefs, length):
    
    coefs = np.array(coefs)
    
    # initial values
    series = [np.random.normal() for i in range(lags)]
     
    for i in range(length):
        pre_vals = np.array(series[-lags:][::-1])
        new_val = np.sum(coefs*pre_vals) + np.random.normal()
        series.append(new_val)
        
    return np.array(series)

# Augmented Dickey-Fuller Test(ADF)
def adf(series):
    result = adfuller(series,maxlag =1, autolag = None)                      # adfuller() with maxlag, take Autoleg = None
    
    return result

#--------------------------------------------------------------------------------------------------------------
# Stationary AR(1) TS
ar_1_s = generate_ar_process(1,[0.5],100)
result = adf(ar_1_s)
print('ADF Test Statistic is {} and p-value is {}'.format(result[0], result[1]))
plt.figure(figsize=(10,4))
plt.plot(ar_1_s)
plt.show()

#--------------------------------------------------------------------------------------------------------------
# Non-Stationary AR(1) TS
ar_1_ns = generate_ar_process(1,[1],100)
result = adf(ar_1_ns)
print('ADF Test Statistic is {} and p-value is {}'.format(result[0], result[1]))
plt.figure(figsize=(10,4))
plt.plot(ar_1_ns)
plt.show()

#--------------------------------------------------------------------------------------------------------------
# Non-Stationary AR(2) TS
ar_2_s = generate_ar_process(1,[0.5,0.3],100)
result = adf(ar_2_s)
print('ADF Test Statistic is {} and p-value is {}'.format(result[0], result[1]))
plt.figure(figsize=(10,4))
plt.plot(ar_2_s)
plt.show()

#--------------------------------------------------------------------------------------------------------------
# Non-Stationary AR(2) TS
ar_2_ns = generate_ar_process(1,[0.7,0.3],100)
result = adf(ar_2_ns)
print('ADF Test Statistic is {} and p-value is {}'.format(result[0], result[1]))
plt.figure(figsize=(10,4))
plt.plot(ar_2_ns)
plt.show()

## From Panda Course 

### Time Series Basics

In [ ]:
path = 'C:\\Users\\Admin\\Pandas Course\\Course_Materials_Part4\\Video_Lecture_NBs\\temp.csv'

temp= pd.read_csv(path, parse_dates=['datetime'], index_col='datetime')   # parse_date - to convert col in datetime object
# OR
temp_1 = pd.read_csv(path)
temp_1.datetime = pd.to_datetime(temp_1.datetime)
temp = temp_1.set_index(keys = 'datetime')

#--------------------------------------------------------------------------------------------------------------
# indexing & slicing on datetime index
# datetime object format  yyyy-mm-dd hh:mm:ss
# df.loc[a:b:c,:] means start label index  a, end label index b with step size c
temp.loc['2015',:]                                                               # all rows in 2015                                            
temp.loc['2015-05',:]                                                            # all rows in May'15                                                  
temp.loc['2015-01-01' : '2015-12-31',:]                                          # all rows in year 2015 
temp.loc['2015-01-01' : '2015-12-31',:].equals(temp.loc['2015',:])               # equals()
temp.loc['2015-01-01' : ,:]                                                      # from 2015-01-01 to end index
temp.loc[:'2015-01-01',:]                                                        # from start to 2015-01-01

two_timesstamps = pd.to_datetime(['2015-05-20 10:00:00','2015-05-20 12:00:00'])  # multiple values filtering
temp.loc[two_timesstamps,:]

#--------------------------------------------------------------------------------------------------------------
# creating customised datetime index using pd.date_range()
# need to define 3 parameters out of 4 (start, end,freq,periods)
# if D/W/M/Q/A means end of the period unless defined with S
pd.date_range(start = '2015-07-01', end = '2015-07-31', freq='D')
pd.date_range(start = '2015-07-01', periods=31, freq='D')
pd.date_range(end = '2015-07-31', periods=31, freq='D')                         # daily frequency
pd.date_range(start = '2015-07-01',end ='2015-07-31',freq='B')                  # business days frequency
pd.date_range(start = '2015-07-01',periods = 16,freq='BM')                      # last business day of month
pd.date_range(start = '2015-07-01',periods = 16,freq='H')                       # hourly frequency
pd.date_range(start = '2015-07-01',periods = 6,freq='W')                        # weekly frequency
pd.date_range(start = '2015-07-01',periods = 6,freq='W-Wed')                    # weekly frequency ending with wednesday
pd.date_range(start = '2015-07-14',periods = 6,freq='M')                        # month end frequency
pd.date_range(start = '2015-07-14',periods = 6,freq='MS')                       # month start frequency so next month since start from '2015-07-14'
pd.date_range(start = '2015-07-14',periods = 6,freq=pd.DateOffset(months =1))   # pd.DateOffset()
pd.date_range(start = '2015-07-14',periods = 6,freq='Q')                        # quarter end frequency
pd.date_range(start = '2015-07-14',periods = 6,freq='QS')                       # quarter start frequency, similar to MS
pd.date_range(start = '2015-07-14',periods = 6,freq='Q-Dec')                    # ending with Dec quarter frequency       
pd.date_range(start = '2015-07-14',periods = 6,freq='QS-Dec')                   # quarter start - Dec, Mar,Jun,Sep    
pd.date_range(start = '2015-07-14',periods = 6,freq='A')                        # annual end frequency
pd.date_range(start = '2015-07-14',periods = 6,freq='AS')                       # annual start frequency
pd.date_range(start = '2015-07-14',periods = 6,freq='AS-Aug')                   # annual start frequency
pd.date_range(start = '2015-07-14',periods = 6,freq=pd.DateOffset(years =1))    # pd.DateOffset()

pd.date_range(start = '2015-07-01',periods = 16,freq='3D8H')                    # frequency - 3 days and 8 hours

#--------------------------------------------------------------------------------------------------------------
# downsampling time series with resample()

# resample() is very similar to groupby() - it creats combination of dataframes based on resampling frequency.
# it creats list of tuple (first element - resampling key, second element - dataframe)
list(temp.resample('D'))[1][1]                      

# resampling without apply()

temp.resample('D').first()                          # first element of every dataframe
temp.resample('D').last()                           # last element of every dataframe
temp.resample('D').mean()                           # mean of every dataframe
temp.resample('MS',loffset = '14D').mean()          # loffset means labelling for each row
temp.resample('Q-Feb').mean()                        

# resampling with apply() - on both dataframe and series

def first_day(df):
    return df.iloc[0,:]

temp.resample(rule = 'D').apply(first_day)                                                       # apply()

temp.resample(rule = 'D').apply(first_day).plot(y = 'Close',figsize =(12,8), kind = 'bar')       # resampling on dataframe
temp.LA.resample(rule = 'D').apply(first_day).plot(figsize =(12,8), kind = 'bar')                # resampling on series
plt.show()

# not preferable
temp_m = temp.resample('M',kind = 'period').mean()  # from timestamp index to period index, better way of labelling each row 
temp_m.to_timestamp(how= 'start')                   # to_timestamp()
 

### Slicing and Indexing on datetime index

In [ ]:
tickers = ['XOM', 'CVX','^GSPC']
prices = yf.download(tickers, start, end, progress = False)
raw_data = prices['Close']

raw_data[10:20]                   # this will work , not the right thing without loc/iloc
raw_data.loc[10:20,:]             # this will not work

# try using iloc while handling with datetime index

### Python for Finance and Investing

In [ ]:
tickers = ['AAPL','BA','KO','IBM','DIS','MSFT']
stocks = yf.download(tickers = tickers, start='2010-01-01', end = '2019-12-31')  # yf.download()
file = stocks.to_csv('C:\\Users\\Admin\\Desktop\\stocks.csv')
stocks = pd.read_csv(file, header = [0,1], index_col=[0], parse_dates=[0])

stocks.columns = stocks.columns.to_flat_index()                                  # to_flat_index() - to flatten col index

close = stocks.loc[:,('Close',)]                                                 # use of multiindex- without slice so single index                                                   
close_norm = close.div(close.iloc[0]).mul(100)                                   # normalization - operation to whole df

# normalize the multiple time series, base =100 for 1st data point of time series

df = close.normalize()                                                           # normalize()                                                         

aapl = close.AAPL.to_frame()
aapl['leg_1'] = aapl.shift(periods=1,axis=0)                                     # shift()
aapl['diff'] = aapl.AAPL.sub(aapl.leg_1)
aapl['per_change'] = aapl.AAPL.div(aapl.leg_1).sub(1).mul(100)
aapl['diff_2'] = aapl.AAPL.diff(periods =1)                                      # diff()
aapl['per_change_2'] = aapl.AAPL.pct_change(periods =1).mul(100)                 # pct_change()
aapl.per_change.equals(aapl.per_change_2)                                        # equals() - equality of dataframe, series

aapl.resample('BM').AAPL.last().pct_change(periods =1).mul(100).to_frame()       # compare it with groupby

#--------------------------------------------------------------------------------------------------------------
# operation on whole dataframe
ret = close.pct_change(periods =1).dropna()
ret_new = ret.describe().transpose().loc[:,['mean','std']]
ret_new.columns = ['mean1','std1']

ret_new.plot(kind = 'scatter', x = 'std1',y ='mean1',figsize =(12,8))
plt.xlabel('standard deviation', fontsize =12)
plt.ylabel('mean', fontsize =12)

for i in range(0,len(ret_new)):
    plt.annotate(text = ret_new.index[i],xy=(ret_new.std1[i]+0.0001,ret_new.mean1[i]+0.000001))
plt.show()

#--------------------------------------------------------------------------------------------------------------
# methods for datetime index
close.index.day                                 # day of the index
close.index.month                               # month of the index
close.index.quarter                             # quarter - (Jan-Mar) is Q1
close.index.year                                # year of the index
close.index.week                                # week of the date in a year
close.index.weekday                             # day, monday starts from 0
close.index.weekofyear                          # same as week
close.index.day_name()                          # day name
close.index.month_name                          # month name
close.index.days_in_month                       # days in that month
close.index.is_month_end                        # to check whether day is month end 

all_days = pd.date_range(start = '2009-12-31', end = '2019-12-30', freq='D')
close_new = close.reindex(all_days)                                              # reindex() - with new indexing schema
close_new['day'] = close_new.index.day_name()
close_new['quarter'] = close_new.index.quarter
close_new.fillna(method = 'bfill', inplace =True)                                # fillna() with 'bfill'/'ffill'/'interpolate'